In [ ]:
import sys
import os
import math
import time
import json
import re
from typing import Any, Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import patches, animation

In [ ]:
# Start from Matplotlib defaults (wipes seaborn/Jupyter styles)
plt.style.use("default")           # or: mpl.rcdefaults()

# If seaborn was used earlier in the session:
try:
    import seaborn as sns
    sns.reset_orig()
except Exception:
    pass

BASE = 12
mpl.rcParams.update({
    "text.usetex": False,
    "mathtext.fontset": "cm",
    "font.family": "serif",
    "font.serif": ["CMU Serif", "Computer Modern", "STIX", "DejaVu Serif"],
    "axes.unicode_minus": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "font.size": BASE,
    "axes.titlesize": BASE * 1.2,
    "axes.labelsize": BASE,
    "xtick.labelsize": BASE * 0.9,
    "ytick.labelsize": BASE * 0.9,
    "legend.fontsize": BASE * 0.9,
    "legend.title_fontsize": BASE,
    "figure.titlesize": BASE * 1.3,

    # lock all text/axis elements to true black
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.titlecolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",

    # avoid transparency/lightening in exports
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
})

# When saving:
# plt.savefig("figure.pdf", bbox_inches="tight", transparent=False)

In [ ]:
print(sys.executable)

## Pitch Dimensions

In [ ]:
# ------------------------------
# Config
# ------------------------------

# Overpass API endpoints (you can add/remove as needed)
OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    # "https://overpass.kumi.systems/api/interpreter",
    # "https://overpass.openstreetmap.fr/api/interpreter",
]

# Be nice and identify your app + contact
HEADERS = {
    "User-Agent": "toppserien-gps-analysis/0.1 (contact: your-email@example.com)"
}

# Your arenas: approx centre of the main match pitch / stadium
ARENAS: Dict[str, Tuple[float, float]] = {
    "Arna Idrettspark": (60.426389, 5.471667),
    "Avaldsnes Idrettssenter": (59.356667, 5.264167),
    "Klepp Stadion": (58.775556, 5.633333),
    "Sofiemyr stadion": (59.800556, 10.818611),
    "LSK-hallen": (59.962222, 11.068889),
    "Kringsjå kunstgress": (59.965556, 10.726944),
    "Røa kunstgress": (59.941809, 10.634959),
    "Koteng Arena": (63.445278, 10.451667),
    "Stemmemyren": (60.425833, 5.305556),
    "Intility Arena": (59.917778, 10.806667),
}


# ------------------------------
# Utility functions
# ------------------------------

def haversine(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Great-circle distance between two points on the Earth (in meters).
    """
    R = 6371000  # Earth radius in meters
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (math.sin(dphi / 2) ** 2 +
         math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c


def build_overpass_query(lat: float, lon: float, radius: int = 250) -> str:
    """
    Build an Overpass QL query to find football/soccer pitches
    within `radius` meters of (lat, lon).
    """
    # We query leisure=pitch with sport=football/soccer,
    # and also plain leisure=pitch as a fallback.
    return f"""
    [out:json][timeout:25];
    (
      way["leisure"="pitch"]["sport"~"football|soccer"](around:{radius},{lat},{lon});
      way["leisure"="pitch"](around:{radius},{lat},{lon});
    );
    out body;
    >;
    out skel qt;
    """


def call_overpass(query: str,
                  max_retries: int = 5,
                  base_sleep: float = 2.0) -> Dict[str, Any]:
    """
    Call Overpass API, trying multiple endpoints, with retries and exponential backoff
    on 429/5xx or network errors.
    """
    last_error: Optional[Exception] = None

    for url in OVERPASS_URLS:
        sleep = base_sleep
        for attempt in range(max_retries):
            try:
                resp = requests.post(
                    url,
                    data={"data": query},
                    headers=HEADERS,
                    timeout=60
                )
                if resp.status_code == 429 or 500 <= resp.status_code < 600:
                    # Rate limit or server error -> backoff and retry
                    print(
                        f"Overpass {url} returned {resp.status_code}. "
                        f"Retrying in {sleep:.1f}s (attempt {attempt+1}/{max_retries})..."
                    )
                    time.sleep(sleep)
                    sleep *= 2
                    continue

                resp.raise_for_status()
                return resp.json()

            except requests.exceptions.RequestException as e:
                last_error = e
                print(
                    f"Error calling {url}: {e}. "
                    f"Retrying in {sleep:.1f}s (attempt {attempt+1}/{max_retries})..."
                )
                time.sleep(sleep)
                sleep *= 2

        print(f"Giving up on {url} after {max_retries} attempts.")

    raise RuntimeError(f"All Overpass endpoints failed. Last error: {last_error}")


def clean_polygon_coords(coords: List[Tuple[float, float]],
                         tol: float = 1e-9) -> List[Tuple[float, float]]:
    """
    Clean a polygon coordinate list from OSM:

    - Remove consecutive duplicate coordinates
    - Drop closing node if it is (almost) identical to the first

    This converts [A, B, C, D, A] into [A, B, C, D].
    """
    if not coords:
        return coords

    cleaned: List[Tuple[float, float]] = [coords[0]]
    for lat, lon in coords[1:]:
        lat0, lon0 = cleaned[-1]
        if abs(lat - lat0) > tol or abs(lon - lon0) > tol:
            cleaned.append((lat, lon))

    # Drop last if same as first
    lat_first, lon_first = cleaned[0]
    lat_last, lon_last = cleaned[-1]
    if abs(lat_first - lat_last) <= tol and abs(lon_first - lon_last) <= tol:
        cleaned = cleaned[:-1]

    return cleaned


# ------------------------------
# Pitch fetching
# ------------------------------

def get_nearest_pitch_polygon(lat: float,
                              lon: float,
                              radius: int = 250) -> Optional[Dict[str, Any]]:
    """
    Query OSM via Overpass and return the nearest pitch polygon
    to (lat, lon), as a dict:

    {
      "id": way_id,
      "tags": {...},
      "coords": [(lat1, lon1), (lat2, lon2), ...]  # cleaned unique corners
    }

    Returns None if nothing is found.
    """
    query = build_overpass_query(lat, lon, radius=radius)
    data = call_overpass(query)

    node_lookup: Dict[int, Tuple[float, float]] = {}
    ways: List[Dict[str, Any]] = []

    for el in data.get("elements", []):
        if el["type"] == "node":
            node_lookup[el["id"]] = (el["lat"], el["lon"])
        elif el["type"] == "way":
            ways.append(el)

    # Build polygons
    polygons: List[Dict[str, Any]] = []
    for way in ways:
        raw_coords: List[Tuple[float, float]] = []
        for nid in way.get("nodes", []):
            if nid in node_lookup:
                raw_coords.append(node_lookup[nid])

        if len(raw_coords) >= 3:
            coords_clean = clean_polygon_coords(raw_coords)
            if len(coords_clean) >= 3:
                polygons.append({
                    "id": way["id"],
                    "tags": way.get("tags", {}),
                    "coords": coords_clean,
                })

    if not polygons:
        return None

    # Pick polygon whose centroid is closest to the input (lat, lon)
    def centroid(poly_coords: List[Tuple[float, float]]) -> Tuple[float, float]:
        lats = [c[0] for c in poly_coords]
        lons = [c[1] for c in poly_coords]
        return (sum(lats) / len(lats), sum(lons) / len(lons))

    best: Optional[Dict[str, Any]] = None
    best_dist = float("inf")

    for poly in polygons:
        clat, clon = centroid(poly["coords"])
        d = haversine(clat, clon, lat, lon)
        if d < best_dist:
            best_dist = d
            best = poly

    return best

def find_nearest_stadium(
    df_or_lat,
    pitches,
    lat_col="lat",
    lon_col="lon",
    stadium_col="stadium_name",
    pitch_lat_col="lat",
    pitch_lon_col="lon",
):
    """
    Find the nearest stadium in `pitches` to the given GPS data.

    Parameters
    ----------
    df_or_lat : DataFrame OR float
        If DataFrame: must contain lat_col/lon_col.
        If float: treated as latitude, and you must pass lon as second argument
        (see overload below).
    pitches : DataFrame
        Stadium table. Must contain stadium_col and either:
          - columns pitch_lat_col/pitch_lon_col giving polygon points per stadium
            (multiple rows per stadium), OR
          - columns 'center_lat','center_lon' per stadium (one row per stadium).
    lat_col/lon_col : str
        Column names in df_or_lat if it's a DataFrame.
    stadium_col : str
        Stadium name column in pitches.

    Returns
    -------
    nearest_name : str
    nearest_dist_m : float
    centers_df : DataFrame (stadium, center_lat, center_lon, dist_m)
    """

    # ---- get representative point (lat0, lon0) ----
    if isinstance(df_or_lat, pd.DataFrame):
        lat0 = pd.to_numeric(df_or_lat[lat_col], errors="coerce").median()
        lon0 = pd.to_numeric(df_or_lat[lon_col], errors="coerce").median()
    else:
        raise ValueError("Pass a DataFrame of samples; this version expects df_or_lat as DataFrame.")

    if np.isnan(lat0) or np.isnan(lon0):
        raise ValueError("Could not compute representative lat/lon from df.")

    # ---- compute stadium centers ----
    if {"center_lat", "center_lon"}.issubset(pitches.columns):
        centers = (
            pitches[[stadium_col, "center_lat", "center_lon"]]
            .drop_duplicates(subset=[stadium_col])
            .rename(columns={"center_lat":"_clat", "center_lon":"_clon"})
        )
    else:
        # assume multiple polygon rows per stadium in pitch_lat_col/pitch_lon_col
        centers = (
            pitches.groupby(stadium_col)[[pitch_lat_col, pitch_lon_col]]
            .mean()
            .reset_index()
            .rename(columns={pitch_lat_col:"_clat", pitch_lon_col:"_clon"})
        )

    # ---- distances ----
    centers["dist_m"] = _haversine_m(lat0, lon0, centers["_clat"], centers["_clon"])

    # nearest
    j = centers["dist_m"].idxmin()
    nearest_name = centers.loc[j, stadium_col]
    nearest_dist_m = float(centers.loc[j, "dist_m"])

    return nearest_name, nearest_dist_m, centers

In [ ]:
# ------------------------------
# Main script
# ------------------------------

def main() -> None:
    all_pitch_polygons: Dict[str, Dict[str, Any]] = {}

    for name, (lat, lon) in ARENAS.items():
        print(f"\nFetching pitch for {name} ({lat}, {lon})...")
        try:
            poly = get_nearest_pitch_polygon(lat, lon, radius=300)
        except RuntimeError as e:
            print(f"  ERROR: {e}")
            continue

        if poly is None:
            print("  No pitch found nearby.")
        else:
            coords = poly["coords"]
            print(
                f"  Found way {poly['id']} with {len(coords)} unique vertices. "
                f"Tags: {poly.get('tags', {})}"
            )
            # Store everything (you can simplify if you only want coords)
            all_pitch_polygons[name] = {
                "way_id": poly["id"],
                "tags": poly.get("tags", {}),
                "coords": coords,
            }

        # Be gentle to Overpass
        time.sleep(2)

    # Save to JSON
    outfile = "toppserien_pitches.json"
    with open(outfile, "w", encoding="utf-8") as f:
        json.dump(all_pitch_polygons, f, indent=2, ensure_ascii=False)

    print(f"\nSaved {len(all_pitch_polygons)} pitch polygons to {outfile}")


if __name__ == "__main__":
    main()

In [ ]:
with open("toppserien_pitches.json", "r", encoding="utf-8") as f:
    pitches = json.load(f)

Stemmemyren_stad = pitches["Stemmemyren"]["coords"]
Stemmemyren_stad

### Match Dates + Scores + Times

In [ ]:
SRC = "kamper.xlsx"   # or "kamper.csv"

# ---------- helpers
def _clean_str(x):
    if pd.isna(x): return ""
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    return s.replace("–","-").replace("—","-")

def _guess_col(df: pd.DataFrame, patterns: List[str]) -> Optional[str]:
    norm = {c: re.sub(r"[^a-z0-9]+", "", str(c).lower()) for c in df.columns}
    for pat in patterns:
        rx = re.compile(pat)
        for c, n in norm.items():
            if rx.search(n): return c
    return None

def _split_match_col(series: pd.Series) -> Tuple[pd.Series, pd.Series]:
    s = series.astype(str).map(_clean_str)
    m = s.str.extract(r"^(?P<home>.+?)\s*[-vV]\s*(?P<away>.+?)$")
    if m is not None and not m.empty:
        return m["home"].fillna(""), m["away"].fillna("")
    return pd.Series([""]*len(s), index=s.index), pd.Series([""]*len(s), index=s.index)

def _excel_serial_to_ts(vals: pd.Series) -> pd.Series:
    s = pd.to_numeric(vals, errors="coerce")
    base = pd.Timestamp("1899-12-30")
    ts = base + pd.to_timedelta(s, unit="D")
    ts[s.isna()] = pd.NaT
    return ts

def _parse_time_like(series: pd.Series) -> pd.Series:
    """
    Return HH:MM strings from textual times, Excel serial fractions, or 15.00 format.
    No pd.to_datetime -> no inference warnings.
    """
    out = pd.Series([""]*len(series), index=series.index, dtype=object)

    # 1) Numeric (Excel fraction of a day)
    num = pd.to_numeric(series, errors="coerce")
    mask_num = num.notna()
    if mask_num.any():
        frac = (num[mask_num] % 1.0)
        secs = np.round(frac * 86400).astype(int)
        hh = (secs // 3600) % 24
        mm = (secs % 3600) // 60
        out.loc[mask_num] = [f"{int(h):02d}:{int(m):02d}" for h, m in zip(hh, mm)]

    # 2) Text forms
    s = series.astype(str)

    # normalize: drop "Kl." and spaces, convert dots to colons
    s_norm = (s.str.replace(r"(?i)^kl\.?\s*", "", regex=True)
                .str.replace(" ", "", regex=False)
                .str.replace(".", ":", regex=False))

    # keep only HH:MM patterns
    m = s_norm.str.extract(r"^\s*(\d{1,2}):(\d{2})\s*$")
    mask_text = m[0].notna() & m[1].notna()
    if mask_text.any():
        h = m.loc[mask_text, 0].astype(int).clip(0, 23)
        mi = m.loc[mask_text, 1].astype(int).clip(0, 59)
        out.loc[mask_text] = [f"{int(H):02d}:{int(M):02d}" for H, M in zip(h, mi)]

    return out

def _ensure_date_time(date_s: pd.Series, time_s: Optional[pd.Series]) -> Tuple[pd.Series, pd.Series]:
    # Dates: try strict parsing with dayfirst, then Excel serials; ffill blanks
    dt_try = pd.to_datetime(date_s, errors="coerce", dayfirst=True)
    if dt_try.isna().mean() > 0.3:
        dt_try = dt_try.fillna(_excel_serial_to_ts(date_s))
    dt_try = dt_try.ffill()
    date_out = dt_try.dt.date.astype(str).where(dt_try.notna(), "")

    # Times: from dedicated column if present; else from date fraction; ffill blanks
    if time_s is not None and time_s.notna().any():
        time_out = _parse_time_like(time_s)
    else:
        tod = dt_try.dt.strftime("%H:%M")
        time_out = tod.mask(tod == "00:00", "")
    time_out = time_out.replace("", np.nan).ffill().fillna("")
    return date_out, time_out

def _ffill_visual_merges(df: pd.DataFrame) -> pd.DataFrame:
    like_date = [c for c in df.columns if re.search(r"(dato|date|kampdato|dag)", str(c), re.I)]
    like_time = [c for c in df.columns if re.search(r"(tid|time|klokkeslett|kickoff|avspark)", str(c), re.I)]
    f = df.copy()
    for c in like_date + like_time:
        f[c] = f[c].replace("", np.nan).ffill()
    return f


# ---------- main loader
def load_matches_table(
    src_path: str = r"C:\Users\[REDACTED-USER]\OneDrive - [REDACTED-EMPLOYER]\repos\UCLAI\data\kamper.xlsx"
) -> pd.DataFrame:

    src = str(src_path).lower()
    if src.endswith((".xlsx",".xls")):
        xls = pd.ExcelFile(src_path)
        frames = []
        for sh in xls.sheet_names:
            try:
                df = pd.read_excel(xls, sh)
            except Exception:
                continue
            if not df.empty:
                frames.append(_ffill_visual_merges(df))
        raw = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    else:
        raw = pd.read_csv(src_path)

    if raw.empty:
        return pd.DataFrame(columns=["date","time","home","away","score"])

    raw = raw.loc[:, raw.notna().any(axis=0)]

    # Guess likely columns
    date_col  = _guess_col(raw, [r"(dato|date|kampdato|dag)$"])
    time_col  = _guess_col(raw, [r"(tid|time|klokkeslett|kickoff|avspark)$"])
    home_col  = _guess_col(raw, [r"(hjemmelag|home|vertslag|vert|lagh|lag1)$"])
    away_col  = _guess_col(raw, [r"(bortelag|away|gjestelag|lagb|lag2)$"])
    match_col = _guess_col(raw, [r"(kamp|match|oppgjor|oppgjr)$"])
    score_col = _guess_col(raw, [r"(resultat|score|sluttresultat)$"])

    date_s = raw[date_col] if date_col else pd.Series([""]*len(raw))
    time_s = raw[time_col] if time_col else None

    if home_col and away_col:
        home = raw[home_col].astype(str).map(_clean_str)
        away = raw[away_col].astype(str).map(_clean_str)
    elif match_col:
        home, away = _split_match_col(raw[match_col])
    else:
        best = None
        for c in raw.columns:
            s = raw[c].astype(str)
            if s.str.contains(r"\s[-–—vV]\s", regex=True).mean() > 0.5:
                best = c; break
        home, away = _split_match_col(raw[best]) if best else (pd.Series([""]*len(raw)), pd.Series([""]*len(raw)))

    # Score "x-y" from many variants
    if score_col:
        sc_txt = raw[score_col].astype(str).map(_clean_str)
    else:
        sc_txt = pd.Series([""]*len(raw))
    sc = sc_txt.str.extract(r"^\s*(\d+)\s*[-:\.]\s*(\d+)\s*$")
    score = (sc[0].fillna("") + "-" + sc[1].fillna("")).str.strip("-")

    date_out, time_out = _ensure_date_time(date_s, time_s)

    out = pd.DataFrame({
        "date": date_out.map(_clean_str),
        "time": time_out.map(_clean_str),
        "home": home.map(_clean_str),
        "away": away.map(_clean_str),
        "score": score.map(_clean_str),
    })
    out = out[(out["home"] != "") | (out["away"] != "")]
    out = out.reset_index(drop=True)
    return out


In [ ]:
# ---------- run & filter to Rosenborg
matches = load_matches_table(SRC)
rosenborg = matches[matches["home"].str.contains("rosenborg", case=False, na=False) |
                    matches["away"].str.contains("rosenborg", case=False, na=False)].reset_index(drop=True)

rosenborg

### Match Level Analytics

In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

import os
print("REGION:", os.getenv("AWS_REGION"))
print("BUCKET:", os.getenv("S3_BUCKET"))


import boto3, s3fs
REGION = os.getenv("AWS_REGION", "eu-west-2")
BUCKET = os.getenv("S3_BUCKET", "ucl-ai-soccormon-dataset")

b3 = boto3.Session(region_name=REGION)
#fs = s3fs.S3FileSystem(session=getattr(b3, "_session", b3))
fs = s3fs.S3FileSystem(anon=False)
print("S3 ready:", REGION)

In [ ]:
prefix = "ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-07/2020-07-25"
hits = [k for k in fs.find(prefix, maxdepth=6) if k.endswith(".parquet")]
print("Found", len(hits), "parquet(s). Showing a few:")
for k in hits:
    print(k)

In [ ]:
# ---------- read parquet (local or via fsspec) ----------
def read_parquet_maybe_fs(path: str, fs: Optional[Any] = None, fs_prefix: Optional[str] = None) -> pd.DataFrame:
    p = str(path)
    if fs_prefix and not p.startswith(fs_prefix):
        p = fs_prefix + p
    if fs is None:
        return pd.read_parquet(p)
    with fs.open(p, "rb") as f:
        return pd.read_parquet(f)

# ---------- pick Nth game (1-based) from Rosenborg df ----------
def pick_game_row(rbk: pd.DataFrame, *, game_number: int = 1) -> pd.Series:
    df = rbk.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    # Sort by date, then time (if present) for stable ordering
    if "time" in df.columns:
        tt = pd.to_datetime(df["time"], errors="coerce")
        df["_time_sort"] = tt.dt.strftime("%H:%M").fillna("00:00")
    else:
        df["_time_sort"] = "00:00"
    df = df[df["date"].notna()].sort_values(["date", "_time_sort"]).reset_index(drop=True)
    if not (1 <= game_number <= len(df)):
        raise IndexError(f"Requested game_number={game_number}, but only {len(df)} dated games found.")
    return df.iloc[game_number - 1]

# ---------- list parquets for that date ----------
def list_parquets_for_one_date(
    fs: Any,
    *,
    date: pd.Timestamp,
    root_template: str = "ucl-ai-soccormon-dataset/data/objective_TEAM_A_{year}",
    maxdepth: int = 6,
    team_tag: Optional[str] = "TeamA",
    filename_contains: Optional[str] = None,
) -> List[str]:
    d = pd.to_datetime(date, errors="coerce")
    if pd.isna(d):
        return []
    year = d.year
    prefix = f"{root_template.format(year=year)}/{year:04d}-{d.month:02d}/{d:%Y-%m-%d}"
    try:
        keys = [k for k in fs.find(prefix, maxdepth=maxdepth) if k.endswith(".parquet")]
    except Exception as e:
        print(f"[warn] fs.find failed for {prefix}: {e}")
        keys = []

    def keep(basename: str) -> bool:
        if team_tag and f"-{team_tag}-" not in basename:
            return False
        if filename_contains and filename_contains not in basename:
            return False
        return True

    return [k for k in keys if keep(os.path.basename(k))]

# ---------- fast category-based time shifter (+/- hours, mod 24) ----------
def _shift_time_categories_fast(s: pd.Series, hours: int) -> pd.Series:
    """
    Shift 'time' strings (HH:MM | HH:MM:SS | HH:MM:SS.mmm) by whole hours (mod 24)
    by remapping ONLY the unique categories. Returns object dtype strings.
    Leaves unparsable categories unchanged.
    """
    if s.dtype == "O" or pd.api.types.is_string_dtype(s):
        cat = s.astype("category")
        cats = pd.Series(cat.cat.categories, dtype="string")
        # parse categories once (tiny array vs huge column)
        m = cats.str.extract(r"^\s*(\d{1,2}):(\d{2})(?::(\d{2})(?:\.(\d{1,6}))?)?\s*$", expand=True)
        ok = m[0].notna() & m[1].notna()

        new_cats = cats.copy()
        if ok.any():
            hh = pd.to_numeric(m.loc[ok, 0], errors="coerce").fillna(0).astype(int)
            mm = pd.to_numeric(m.loc[ok, 1], errors="coerce").fillna(0).astype(int)
            ss = pd.to_numeric(m.loc[ok, 2], errors="coerce").fillna(0).astype(int)
            fr = m.loc[ok, 3].fillna("")

            hh = (hh + int(hours)) % 24
            base = hh.astype(str).str.zfill(2) + ":" + mm.astype(str).str.zfill(2)

            has_sec = m.loc[ok, 2].notna()
            base.loc[has_sec] = base.loc[has_sec] + ":" + ss.loc[has_sec].astype(int).astype(str).str.zfill(2)
            has_frac = fr.str.len() > 0
            if has_frac.any():
                idx = fr.index[has_frac]
                base.loc[idx] = base.loc[idx] + "." + fr.loc[idx]

            new_cats.loc[ok] = base

        out_cat = pd.Categorical.from_codes(cat.cat.codes, new_cats)
        return pd.Series(out_cat.astype("object"), index=s.index)
    else:
        # Non-string 'time' (rare). If datetime-like, add hours;
        # otherwise just return unchanged.
        if pd.api.types.is_datetime64_any_dtype(s):
            return (s + pd.Timedelta(hours=hours)).astype("datetime64[ns]")
        return s

# ---------- loader: minimal + fast +2h time shift ----------
def load_one_game_parquets_shift_time_fast(
    rbk: pd.DataFrame,
    *,
    game_number: int = 1,                 # 1-based
    fs: Any = None,
    fs_prefix: Optional[str] = "s3://",
    team_tag: Optional[str] = "TeamA",
    filename_contains: Optional[str] = None,
    root_template: str = "ucl-ai-soccormon-dataset/data/objective_TEAM_A_{year}",
    maxdepth: int = 6,
    max_workers: int = 8,
    shift_hours: int = 2,                 # <--- fast shift applied after concat
) -> Tuple[pd.DataFrame, List[str], Dict[str, str]]:
    row = pick_game_row(rbk, game_number=game_number)
    date = pd.to_datetime(row["date"])
    meta = {
        "date": str(date.date()),
        "time": str(row.get("time", "")),
        "home": str(row.get("home", "")),
        "away": str(row.get("away", "")),
        "score": str(row.get("score", row.get("result", ""))),
    }

    keys = list_parquets_for_one_date(
        fs, date=date, root_template=root_template, maxdepth=maxdepth,
        team_tag=team_tag, filename_contains=filename_contains
    )
    if not keys:
        print(f"[info] No parquets found for {date.date()} (team_tag={team_tag!r}).")
        return pd.DataFrame(), [], meta

    try:
        from tqdm.auto import tqdm
        iterator = True
    except Exception:
        iterator = False
        tqdm = lambda x, **k: x  # type: ignore

    def _load_one(p: str) -> pd.DataFrame:
        return read_parquet_maybe_fs(p, fs=fs, fs_prefix=fs_prefix)

    dfs: List[pd.DataFrame] = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(_load_one, p): p for p in keys}
        it = as_completed(futs)
        if iterator:
            it = tqdm(it, total=len(futs), desc="Files")
        for fut in it:
            try:
                dfp = fut.result()
                if not dfp.empty:
                    dfs.append(dfp)
            except Exception as e:
                print(f"[skip] {futs[fut]}: {e}")

    big = pd.concat(dfs, ignore_index=True, sort=False) if dfs else pd.DataFrame()

    # FAST time shift on the concatenated column
    if shift_hours and "time" in big.columns:
        big = big.copy()
        big["time"] = _shift_time_categories_fast(big["time"], shift_hours)

    return big, keys, meta

In [ ]:
gnum = 9
df_raw, keys, meta = load_one_game_parquets_shift_time_fast(
    rosenborg, game_number=gnum, fs=fs, fs_prefix="s3://", team_tag="TeamA", shift_hours=2
)
print(meta)
print(df_raw["time"].head())  # should reflect +2h shift

In [ ]:
df_raw.head()

In [ ]:
def ensure_timestamp_fast(df: pd.DataFrame, time_col="timestamp") -> pd.DataFrame:
    if time_col in df.columns and pd.api.types.is_datetime64_any_dtype(df[time_col]):
        return df
    out = df.copy()
    if time_col in out.columns:
        out[time_col] = pd.to_datetime(out[time_col], errors="coerce", utc=False)
        return out
    if "time" in out.columns:
        s = out["time"].astype(str)
        ts = pd.to_datetime(s, format="%H:%M:%S.%f", errors="coerce")
        m = ts.isna()
        if m.any():
            ts.loc[m] = pd.to_datetime(s[m], format="%H:%M:%S", errors="coerce")
        base = pd.Timestamp("1970-01-01")
        out[time_col] = base + (ts - ts.dt.normalize())
        return out
    raise KeyError(f"Could not find a usable time column ('{time_col}' or 'time').")

def downsample_to_1hz_fast(df: pd.DataFrame,
                           player_col="player_name",
                           time_col="timestamp",
                           keep="first",              # or "last"
                           assume_sorted=True) -> pd.DataFrame:
    if player_col not in df.columns:
        raise KeyError(f"Missing player column '{player_col}' in df.")
    df_ts = ensure_timestamp_fast(df, time_col=time_col)

    # --- convert to NumPy arrays (avoid pandas alignment) ---
    p_codes = pd.factorize(df_ts[player_col], sort=False)[0].astype(np.int64, copy=False)
    t_ns = df_ts[time_col].astype("int64", copy=False).to_numpy()
    t_sec = (t_ns // 1_000_000_000).astype(np.int64, copy=False)

    if not assume_sorted:
        # sort by (player, second, timestamp)
        order = np.lexsort((t_ns, t_sec, p_codes))
        p_sorted = p_codes[order]
        s_sorted = t_sec[order]

        if keep == "first":
            keep_mask_sorted = np.empty_like(p_sorted, dtype=bool)
            keep_mask_sorted[0] = True
            keep_mask_sorted[1:] = (p_sorted[1:] != p_sorted[:-1]) | (s_sorted[1:] != s_sorted[:-1])
        else:
            keep_mask_sorted = np.empty_like(p_sorted, dtype=bool)
            keep_mask_sorted[-1] = True
            keep_mask_sorted[:-1] = (p_sorted[:-1] != p_sorted[1:]) | (s_sorted[:-1] != s_sorted[1:])

        kept_idx = df_ts.index.to_numpy()[order][keep_mask_sorted]
        return df_ts.loc[np.sort(kept_idx)]

    # fast path: already sorted by player then time
    p = p_codes
    s = t_sec
    if keep == "first":
        keep_mask = np.empty(len(df_ts), dtype=bool)
        keep_mask[0] = True
        keep_mask[1:] = (p[1:] != p[:-1]) | (s[1:] != s[:-1])
    else:
        keep_mask = np.empty(len(df_ts), dtype=bool)
        keep_mask[-1] = True
        keep_mask[:-1] = (p[:-1] != p[1:]) | (s[:-1] != s[1:])
    return df_ts.loc[keep_mask]


In [ ]:
df_1hz = downsample_to_1hz_fast(
    df_raw,
    player_col="player_name",
    time_col="timestamp",
    keep="first",
    assume_sorted=True     # set False if you're unsure your rows are sorted
)
print(len(df_raw), "→", len(df_1hz))


In [ ]:
df_1hz

In [ ]:
def _auto_scale_degrees(lat_raw, lon_raw):
    """
    Ensure lat/lon are in decimal degrees.

    If values look like 'microdegrees' (e.g. ~5.994e7 for 59.94°),
    we divide by 1e6. Otherwise we leave them alone.

    Returns
    -------
    lat_deg, lon_deg, scale
      where scale is the divisor applied to the original numbers.
    """
    lat = pd.to_numeric(lat_raw, errors="coerce")
    lon = pd.to_numeric(lon_raw, errors="coerce")

    # typical magnitude of latitude
    typical = np.nanmedian(np.abs(lat))

    # crude heuristic: if it's way bigger than 180, assume microdegrees
    if typical > 180.0:
        scale = 1e6
        lat = lat / scale
        lon = lon / scale
    else:
        scale = 1.0

    return lat, lon, scale

def attach_xy_from_pitch(
    df_1hz: pd.DataFrame,
    center_latlon,
    R: np.ndarray,
    *,
    lat_col: str = "lat",
    lon_col: str = "lon",
    stadium_name: str | None = None,
    copy: bool = True
) -> pd.DataFrame:
    """Adds x_m, y_m in metres (centre+rotated). Drops invalid lat/lon rows."""
    df = df_1hz.copy() if copy else df_1hz

    lat_raw = pd.to_numeric(df[lat_col], errors="coerce")
    lon_raw = pd.to_numeric(df[lon_col], errors="coerce")
    lat_deg, lon_deg, _ = _auto_scale_degrees(lat_raw, lon_raw)

    bad = ((lat_deg == 0) & (lon_deg == 0)) | (lat_deg.abs() > 90) | (lon_deg.abs() > 180)
    ok = (~bad) & lat_deg.notna() & lon_deg.notna()
    if not ok.any():
        return df.iloc[0:0].copy()

    df = df.loc[ok].reset_index(drop=True)

    # _latlon_to_xy returns NumPy arrays already
    x_local, y_local = _latlon_to_xy(lat_deg.loc[ok], lon_deg.loc[ok], center_latlon[0], center_latlon[1])

    # Ensure 2D array then rotate
    XY_local = np.column_stack((np.asarray(x_local, dtype=float),
                                np.asarray(y_local, dtype=float)))
    XY = XY_local @ R

    df["x_m"] = XY[:, 0]
    df["y_m"] = XY[:, 1]
    if stadium_name:
        df["stadium_name"] = stadium_name
    return df


In [ ]:
# --- tiny helpers (self-contained) ---
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlmb/2)**2
    return 2*R*math.atan2(math.sqrt(a), math.sqrt(1-a))

def polygon_latlon_centroid(coords):
    lats = [float(c[0]) for c in coords]
    lons = [float(c[1]) for c in coords]
    return sum(lats)/len(lats), sum(lons)/len(lons)

def _latlon_to_xy(lat_deg, lon_deg, lat0, lon0):
    m_per_deg_lat = 111_320.0
    m_per_deg_lon = 111_320.0 * math.cos(math.radians(lat0))
    lat_arr = np.asarray(lat_deg, dtype=float)
    lon_arr = np.asarray(lon_deg, dtype=float)
    x_m = (lon_arr - lon0) * m_per_deg_lon
    y_m = (lat_arr - lat0) * m_per_deg_lat
    return x_m, y_m

def polygon_latlon_to_xy(coords, origin_lat, origin_lon):
    lats = np.array([float(c[0]) for c in coords], dtype=float)
    lons = np.array([float(c[1]) for c in coords], dtype=float)
    x_m, y_m = _latlon_to_xy(lats, lons, origin_lat, origin_lon)
    return np.column_stack((x_m, y_m))

# --- NEW: calibrate & rotate the pitch polygon ---
def calibrate_pitch_from_df(df_1hz, pitches, lat_col="lat", lon_col="lon"):
    """
    Finds the nearest stadium, computes pitch centre, and returns:
        stadium_name, (centre_lat, centre_lon), R (2x2), pitch_xy_rot (list of (x,y))
    where pitch_xy_rot is in metres, centred at (0,0), and rotated so the long axis is horizontal.
    """
    lat = pd.to_numeric(df_1hz[lat_col], errors="coerce")
    lon = pd.to_numeric(df_1hz[lon_col], errors="coerce")
    valid = lat.notna() & lon.notna() & (lat != 0) & (lon != 0)
    if not valid.any():
        raise ValueError("No valid lat/lon rows in df_1hz for calibration.")

    lat0 = float(lat.loc[valid].iloc[0])
    lon0 = float(lon.loc[valid].iloc[0])

    # 1) nearest stadium by centroid
    best_name, best_info, best_d = None, None, float("inf")
    for name, info in pitches.items():
        coords = info["coords"]  # list of [lat, lon]
        c_lat, c_lon = polygon_latlon_centroid(coords)
        d = haversine(lat0, lon0, c_lat, c_lon)
        if d < best_d:
            best_name, best_info, best_d = name, info, d

    if best_info is None:
        raise RuntimeError("Could not match any stadium from the pitches dictionary.")

    # 2) centre (lat/lon) and raw polygon -> metres relative to centre
    pitch_coords_ll = best_info["coords"]
    c_lat, c_lon = polygon_latlon_centroid(pitch_coords_ll)
    P = polygon_latlon_to_xy(pitch_coords_ll, c_lat, c_lon)  # metres, origin at pitch centre

    # 3) PCA/SVD to align long axis horizontally
    C = P - P.mean(axis=0)   # (should already be near 0,0; this just stabilizes SVD)
    _, _, Vt = np.linalg.svd(C, full_matrices=False)
    v0, v1 = Vt[0], Vt[1]    # principal axes (unit vectors)
    if v0[0] < 0:            # enforce +x pointing to the "right"
        v0, v1 = -v0, -v1
    R = np.column_stack((v0, v1))  # 2x2 rotation matrix

    # 4) rotate the polygon
    pitch_xy_rot = (P @ R)
    pitch_xy_rot_list = [tuple(pt) for pt in pitch_xy_rot]

    print(f"[calibrate] Stadium='{best_name}', centre=({c_lat:.6f}, {c_lon:.6f}), "
          f"first-sample distance ≈ {best_d:.1f} m")

    return best_name, (c_lat, c_lon), R, pitch_xy_rot_list


In [ ]:
stadium, center_latlon, R, pitch_xy = calibrate_pitch_from_df(df_1hz, pitches, lat_col="lat", lon_col="lon")
df_xy = attach_xy_from_pitch(df_1hz, center_latlon, R, lat_col="lat", lon_col="lon", stadium_name=stadium)

In [ ]:
df_xy

In [ ]:
mpl.rcParams["animation.html"] = "jshtml"  # render inline in notebooks


def _choose_time(df):
    if "timestamp" in df.columns:
        return pd.to_datetime(df["timestamp"], errors="coerce")
    if "time" in df.columns:
        return pd.to_datetime(df["time"], errors="coerce")
    raise ValueError("df must have 'timestamp' or 'time'.")

def draw_pitch(ax, pitch_xy, margin_m=6.0, show_outer_box=True):
    """
    Draw a simple full pitch from pitch_xy polygon, plus an optional
    outer rectangle at the far limits of the simulation (pitch + margin_m).
    """
    P = np.asarray(pitch_xy, dtype=float)
    xmin, ymin = P.min(axis=0)
    xmax, ymax = P.max(axis=0)

    # main pitch outline
    ax.add_patch(
        patches.Polygon(
            P, closed=True, fill=False, lw=2.0, ec="black", zorder=1
        )
    )

    midx = 0.5 * (xmin + xmax)
    midy = 0.5 * (ymin + ymax)

    # halfway line + centre circle/spot
    ax.plot([midx, midx], [ymin, ymax], lw=1.2, color="black", zorder=1)
    ax.add_patch(patches.Circle((midx, midy), 9.15, fill=False, ec="black", lw=1.0, zorder=1))
    ax.add_patch(patches.Circle((midx, midy), 0.2, color="black", zorder=1))

    # penalty & goal boxes (assume goals at xmin/xmax)
    for side in ("left", "right"):
        gx0 = xmin if side == "left" else xmax
        sgn = +1 if side == "left" else -1

        # penalty area
        ax.add_patch(
            patches.Rectangle(
                (min(gx0, gx0 + sgn * 16.5), midy - 40.32 / 2),
                16.5,
                40.32,
                fill=False,
                ec="black",
                lw=1.0,
                zorder=1,
            )
        )

        # 6-yard box
        ax.add_patch(
            patches.Rectangle(
                (min(gx0, gx0 + sgn * 5.5), midy - 18.32 / 2),
                5.5,
                18.32,
                fill=False,
                ec="black",
                lw=1.0,
                zorder=1,
            )
        )

        # penalty spot
        ax.add_patch(patches.Circle((gx0 + sgn * 11.0, midy), 0.2, color="black", zorder=1))

    # ---- outer simulation box (pitch + margin) ----
    outer_xmin = xmin - margin_m
    outer_xmax = xmax + margin_m
    outer_ymin = ymin - margin_m
    outer_ymax = ymax + margin_m

    if show_outer_box:
        ax.add_patch(
            patches.Rectangle(
                (outer_xmin, outer_ymin),
                outer_xmax - outer_xmin,
                outer_ymax - outer_ymin,
                fill=False,
                ec="black",   # thick black
                lw=2.5,       # thicker line
                ls="-",       # solid
                zorder=0.5,
            )
        )

    # aesthetics
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(outer_xmin, outer_xmax)
    ax.set_ylim(outer_ymin, outer_ymax)
    ax.set_facecolor("white")

    for sp in ax.spines.values():
        sp.set_visible(False)

    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

def _prep_positions(df_xy, player_col, x_col, y_col, step_s, pad_limit_s):
    df = df_xy[[player_col, x_col, y_col]].copy()
    df["_ts"] = _choose_time(df_xy)
    df = df.dropna(subset=["_ts"]).sort_values("_ts")

    if df.empty:
        raise ValueError("No valid timestamps after parsing 'time'/'timestamp'.")

    # global time bounds: first → last sample across all players
    t0 = df["_ts"].min().floor("s")
    t1 = df["_ts"].max().ceil("s")
    frames = pd.date_range(t0, t1, freq=f"{int(step_s)}s")

    players = df[player_col].dropna().unique().tolist()
    nF, nP = len(frames), len(players)
    pos = np.full((nF, nP, 2), np.nan, dtype=float)

    for j, pid in enumerate(players):
        g = df[df[player_col] == pid].dropna(subset=[x_col, y_col])
        if g.empty:
            continue

        s = g.set_index("_ts")[[x_col, y_col]].sort_index()
        r = s.reindex(frames, method="pad")

        # how long since last *real* sample?
        last_real = s.index.to_series().reindex(frames, method="pad")
        age = (
            frames.to_series().reset_index(drop=True) -
            last_real.reset_index(drop=True)
        ).dt.total_seconds().to_numpy()

        # blank out if too old
        r.loc[age > float(pad_limit_s), [x_col, y_col]] = np.nan

        pos[:, j, :] = r[[x_col, y_col]].to_numpy()

    return frames.to_numpy(), players, pos


def animate_players_simple(
    df_xy, pitch_xy,
    player_col="player_name", x_col="x_m", y_col="y_m",
    step_s=1, pad_limit_s=2, margin_m=6.0, fps=25
):
    """
    Returns matplotlib.animation.FuncAnimation.

    - Global clock from earliest to latest sample (across all players).
    - Clock shows actual data time (HH:MM:SS) from 'time' or 'timestamp'.
    """
    frames, players, pos = _prep_positions(df_xy, player_col, x_col, y_col, step_s, pad_limit_s)
    nF, nP = pos.shape[0], pos.shape[1]
    if nP == 0 or nF == 0:
        raise ValueError("No frames or players to animate (check your df_xy).")

    fig, ax = plt.subplots(figsize=(10, 6))
    draw_pitch(ax, pitch_xy, margin_m=margin_m)

    # Stable per-player colors
    cmap = plt.get_cmap("tab20").colors
    colors = [cmap[i % len(cmap)] for i in range(nP)]

    # Start with NaNs but correct number of points for colors
    offsets_init = np.full((nP, 2), np.nan)
    scat = ax.scatter(
        offsets_init[:, 0], offsets_init[:, 1],
        s=40, c=colors, edgecolors="black", linewidths=0.6, zorder=3
    )

    # --- CLOCK: put it in axes coordinates, just above the pitch ---
    clock_txt = ax.text(
        0.5, 1.02, "",               # centred, slightly above top of axes
        transform=ax.transAxes,
        ha="center", va="bottom",
        fontsize=14, family="monospace",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.8),
        zorder=10,
        clip_on=False
    )

    def init():
        scat.set_offsets(offsets_init)
        clock_txt.set_text("")
        return scat, clock_txt

    def update(i):
        XY = pos[i]  # (nP, 2)
        scat.set_offsets(XY)
        # frames[i] is already a datetime64[ns]
        clock_txt.set_text(pd.to_datetime(frames[i]).strftime("%H:%M:%S"))
        return scat, clock_txt

    ani = animation.FuncAnimation(
        fig, update, init_func=init,
        frames=nF, interval=1000 / fps,
        blit=False      # <--- important so the clock updates reliably
    )

    # prevent the static figure from showing under the JSHTML animation
    plt.close(fig)

    return ani

In [ ]:
ani = animate_players_simple(
    df_xy, pitch_xy,
    player_col="player_name",
    x_col="x_m", y_col="y_m",
    step_s=30,
    pad_limit_s=2,
    margin_m=6.0,
    fps=25
)
ani

In [ ]:
def _choose_time_col(df):
    if "timestamp" in df.columns:
        tcol = "timestamp"
    elif "time" in df.columns:
        tcol = "time"
    else:
        raise ValueError("df_xy must contain either 'timestamp' or 'time' column.")
    t = pd.to_datetime(df[tcol], errors="coerce")
    if t.isna().all():
        raise ValueError(f"Could not parse any datetimes from column '{tcol}'.")
    return tcol, t


def _compute_depth_from_edges(df_xy, pitch_xy, x_col="x_m", y_col="y_m"):
    """
    For each sample, compute depth (>=0) inside the true pitch:
    min distance to any of the 4 pitch lines, 0 if outside the rectangle.
    """
    P = np.asarray(pitch_xy, dtype=float)
    xmin, ymin = P.min(axis=0)
    xmax, ymax = P.max(axis=0)

    x = df_xy[x_col].to_numpy(dtype=float)
    y = df_xy[y_col].to_numpy(dtype=float)

    dx_left   = x - xmin
    dx_right  = xmax - x
    dy_bottom = y - ymin
    dy_top    = ymax - y

    depth = np.minimum.reduce([dx_left, dx_right, dy_bottom, dy_top])
    depth = np.where(depth < 0, 0.0, depth)  # 0 outside pitch

    return pd.Series(depth, index=df_xy.index, name="depth_from_edge")


def label_active_players(
    df_xy,
    pitch_xy,
    player_col="player_name",
    x_col="x_m",
    y_col="y_m",
    active_depth_m=3.0,      # metres inside from nearest line to count as "active zone"
    activate_s=60.0,         # X: seconds continuous in active zone to go bench -> active
    bench_off_s=120.0,       # X2: seconds continuous OFF pitch to go active -> bench
    on_pitch_eps_m=0.1,      # >= this depth counts as "on pitch"
    label_col="player_status",
):
    """
    Add a per-row label 'active' / 'bench' to df_xy using hysteresis:

      - Bench -> Active:
          player must have been in the active zone (depth_from_edge >= active_depth_m)
          continuously for at least `activate_s` seconds.

      - Active -> Bench:
          player must have been OFF the pitch (depth_from_edge < on_pitch_eps_m)
          continuously for at least `bench_off_s` seconds.

    Between these thresholds, the current status is kept (hysteresis).
    """
    if player_col not in df_xy.columns:
        raise ValueError(f"df_xy must contain a '{player_col}' column.")

    df = df_xy.copy()

    # --- time handling ---
    time_col, t = _choose_time_col(df)
    df[time_col] = t
    df = df.dropna(subset=[time_col])
    if df.empty:
        raise ValueError("No valid time values after parsing 'time'/'timestamp'.")

    # --- depth & flags ---
    df["depth_from_edge"] = _compute_depth_from_edges(df, pitch_xy, x_col=x_col, y_col=y_col)

    # on pitch if inside the true pitch by at least on_pitch_eps_m
    df["on_pitch"] = df["depth_from_edge"] >= on_pitch_eps_m
    # in active zone if deep enough inside
    df["in_active_zone"] = df["depth_from_edge"] >= active_depth_m

    # --- sort & per-player time deltas ---
    df = df.sort_values([player_col, time_col])

    dt_s = (
        df.groupby(player_col)[time_col]
          .diff()
          .dt.total_seconds()
          .fillna(0.0)
          .clip(lower=0.0)
    )
    df["_dt_s"] = dt_s

    # --- hysteresis status per player ---
    df[label_col] = "bench"  # initial state

    for pid, idx in df.groupby(player_col, sort=False).groups.items():
        g = df.loc[idx]

        in_active = g["in_active_zone"].to_numpy()
        on_pitch  = g["on_pitch"].to_numpy()
        dts       = g["_dt_s"].to_numpy()

        status_arr = np.empty(len(g), dtype=object)
        current_status = "bench"
        time_in_active = 0.0
        time_off_pitch = 0.0

        for k in range(len(g)):
            dt = dts[k]

            if current_status == "bench":
                # grow/reset active-zone timer
                if in_active[k]:
                    time_in_active += dt
                else:
                    time_in_active = 0.0

                # bench -> active transition
                if time_in_active >= activate_s:
                    current_status = "active"
                    time_off_pitch = 0.0  # reset off-pitch timer when they become active

            else:  # current_status == "active"
                # grow/reset off-pitch timer
                if not on_pitch[k]:
                    time_off_pitch += dt
                else:
                    time_off_pitch = 0.0

                # active -> bench transition
                if time_off_pitch >= bench_off_s:
                    current_status = "bench"
                    time_in_active = 0.0  # must build up active time again

            status_arr[k] = current_status

        df.loc[idx, label_col] = status_arr

    # --- cleanup ---
    df = df.drop(columns=["_dt_s"])

    return df

In [ ]:
gametime = rosenborg['time'][gnum-1]

active_depth = 6.0    # metres inside pitch for "active zone"
X  = 70.0             # 60 s continuous in active zone to activate
X2 = 110            # 120 s (= 2 min) off pitch to deactivate

df_labeled = label_active_players(
    df_xy,
    pitch_xy,
    player_col="player_name",
    x_col="x_m",
    y_col="y_m",
    active_depth_m=active_depth,
    activate_s=X,
    bench_off_s=X2,
    on_pitch_eps_m=0.1,
    label_col="player_status"
)

df_labeled.head()

In [ ]:
# assumes your existing _choose_time(df) and draw_pitch(ax, pitch_xy, ...) are already defined


def _prep_positions_with_status(df_xy, player_col, x_col, y_col, status_col,
                                step_s, pad_limit_s):
    """
    Align all players to a single global timeline and return:
      frames: np.array of datetimes (global clock)
      players: list of player ids
      pos:   (nF, nP, 2) positions in metres
      status:(nF, nP)    status strings (e.g. 'active' / 'bench')
    """
    # keep only what we need
    df = df_xy[[player_col, x_col, y_col, status_col]].copy()
    df["_ts"] = _choose_time(df_xy)
    df = df.dropna(subset=["_ts"]).sort_values("_ts")

    if df.empty:
        raise ValueError("No valid timestamps after parsing 'time'/'timestamp'.")

    # global time bounds: first → last sample across all players
    t0 = df["_ts"].min().floor("s")
    t1 = df["_ts"].max().ceil("s")
    frames = pd.date_range(t0, t1, freq=f"{int(step_s)}s")

    players = df[player_col].dropna().unique().tolist()
    nF, nP = len(frames), len(players)

    pos = np.full((nF, nP, 2), np.nan, dtype=float)
    status = np.full((nF, nP), "", dtype=object)

    for j, pid in enumerate(players):
        g = df[df[player_col] == pid].dropna(subset=["_ts"])
        if g.empty:
            continue

        # index by time, with status
        s = g.set_index("_ts")[[x_col, y_col, status_col]].sort_index()

        # pad to global frames
        r = s.reindex(frames, method="pad")

        # age since last *real* sample -> blank if too old
        last_real = s.index.to_series().reindex(frames, method="pad")
        age = (
            frames.to_series().reset_index(drop=True)
            - last_real.reset_index(drop=True)
        ).dt.total_seconds().to_numpy()

        r.loc[age > float(pad_limit_s), [x_col, y_col]] = np.nan

        pos[:, j, :] = r[[x_col, y_col]].to_numpy()
        status[:, j] = r[status_col].to_numpy()

    return frames.to_numpy(), players, pos, status


In [ ]:
def draw_pitch(
    ax,
    pitch_xy,
    margin_m=6.0,
    show_outer_box=True,
    active_depth_m=None,        # metres from lines defining the active zone
    show_active_zone=True,
    show_scale_bar=True,
    scale_bar_length_m=10.0,    # length of the scale bar in metres
):
    """
    Draw a football pitch from pitch_xy plus:
      - outer rectangle at the simulation limits (pitch + margin_m)
      - red ring (bench strip) between pitch perimeter and active zone
      - light-green filled active zone (depth >= active_depth_m)
      - horizontal scale bar (in metres).

    Pitch lines are always drawn on top of the coloured areas.
    """
    P = np.asarray(pitch_xy, dtype=float)
    xmin, ymin = P.min(axis=0)
    xmax, ymax = P.max(axis=0)

    # ---- outer simulation box (pitch + margin) ----
    outer_xmin = xmin - margin_m
    outer_xmax = xmax + margin_m
    outer_ymin = ymin - margin_m
    outer_ymax = ymax + margin_m

    if show_outer_box:
        ax.add_patch(
            patches.Rectangle(
                (outer_xmin, outer_ymin),
                outer_xmax - outer_xmin,
                outer_ymax - outer_ymin,
                fill=False,
                ec="black",
                lw=2.5,
                ls="-",
                zorder=0.5,
            )
        )

    # ---- coloured pitch areas (UNDER the lines) ----
    if show_active_zone and active_depth_m is not None and active_depth_m > 0:
        pitch_width  = xmax - xmin
        pitch_height = ymax - ymin

        # 1) Fill whole pitch in light red (bench strip + active zone)
        ax.add_patch(
            patches.Rectangle(
                (xmin, ymin),
                pitch_width,
                pitch_height,
                facecolor="red",
                edgecolor="none",
                alpha=0.1,
                zorder=0.6,  # under lines
            )
        )

        # 2) Overlay the inner active zone in light green
        inner_xmin = xmin + active_depth_m
        inner_xmax = xmax - active_depth_m
        inner_ymin = ymin + active_depth_m
        inner_ymax = ymax - active_depth_m

        if inner_xmax > inner_xmin and inner_ymax > inner_ymin:
            ax.add_patch(
                patches.Rectangle(
                    (inner_xmin, inner_ymin),
                    inner_xmax - inner_xmin,
                    inner_ymax - inner_ymin,
                    facecolor="green",
                    edgecolor="none",
                    alpha=0.3,
                    zorder=0.7,  # still under lines, over red
                )
            )

    # ---- main pitch outline + lines (ON TOP of fills) ----
    # pitch border
    ax.add_patch(
        patches.Polygon(
            P, closed=True, fill=False, lw=2.0, ec="black", zorder=1.5
        )
    )

    midx = 0.5 * (xmin + xmax)
    midy = 0.5 * (ymin + ymax)

    # halfway line
    ax.plot([midx, midx], [ymin, ymax], lw=1.2, color="black", zorder=1.5)

    # centre circle + spot
    ax.add_patch(
        patches.Circle((midx, midy), 9.15, fill=False, ec="black", lw=1.0, zorder=1.5)
    )
    ax.add_patch(
        patches.Circle((midx, midy), 0.2, color="black", zorder=1.5)
    )

    # penalty & goal boxes (assume goals at xmin/xmax)
    for side in ("left", "right"):
        gx0 = xmin if side == "left" else xmax
        sgn = +1 if side == "left" else -1

        # penalty area
        ax.add_patch(
            patches.Rectangle(
                (min(gx0, gx0 + sgn * 16.5), midy - 40.32 / 2),
                16.5,
                40.32,
                fill=False,
                ec="black",
                lw=1.0,
                zorder=1.5,
            )
        )

        # 6-yard box
        ax.add_patch(
            patches.Rectangle(
                (min(gx0, gx0 + sgn * 5.5), midy - 18.32 / 2),
                5.5,
                18.32,
                fill=False,
                ec="black",
                lw=1.0,
                zorder=1.5,
            )
        )

        # penalty spot
        ax.add_patch(
            patches.Circle((gx0 + sgn * 11.0, midy), 0.2, color="black", zorder=1.5)
        )

    # ---- scale bar (in metres) ----
    if show_scale_bar and scale_bar_length_m is not None and scale_bar_length_m > 0:
        pitch_width = xmax - xmin
        length = min(scale_bar_length_m, 0.8 * pitch_width)

        # put it in the lower margin, centered horizontally
        bar_y = outer_ymin + 0.5 * (ymin - outer_ymin)
        bar_x_center = 0.5 * (xmin + xmax)
        bar_x0 = bar_x_center - length / 2.0
        bar_x1 = bar_x_center + length / 2.0

        ax.plot([bar_x0, bar_x1], [bar_y, bar_y], lw=2.0, color="black", zorder=2)
        tick_h = (ymin - outer_ymin) * 0.08
        ax.plot([bar_x0, bar_x0], [bar_y - tick_h, bar_y + tick_h], lw=1.5, color="black", zorder=2)
        ax.plot([bar_x1, bar_x1], [bar_y - tick_h, bar_y + tick_h], lw=1.5, color="black", zorder=2)

        ax.text(
            bar_x_center,
            bar_y - 1.5 * tick_h,
            f"{int(round(length))} m",
            ha="center",
            va="top",
            fontsize=9,
            zorder=2,
        )

    # aesthetics
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(outer_xmin, outer_xmax)
    ax.set_ylim(outer_ymin, outer_ymax)
    ax.set_facecolor("white")

    for sp in ax.spines.values():
        sp.set_visible(False)

    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

In [ ]:
def _prep_positions_with_status(df_xy, player_col, x_col, y_col, status_col,
                                step_s, pad_limit_s):
    """
    Align all players to a single global timeline and return:
      frames: np.array of datetimes (global clock)
      players: list of player ids
      pos:   (nF, nP, 2) positions in metres
      status:(nF, nP)    status strings (e.g. 'active' / 'bench')
    """
    # Keep only what we need
    df = df_xy[[player_col, x_col, y_col, status_col]].copy()
    df["_ts"] = _choose_time(df_xy)
    df = df.dropna(subset=["_ts"]).sort_values("_ts")

    if df.empty:
        raise ValueError("No valid timestamps after parsing 'time'/'timestamp'.")

    # Global time bounds: first → last sample across all players
    t0 = df["_ts"].min().floor("s")
    t1 = df["_ts"].max().ceil("s")
    frames = pd.date_range(t0, t1, freq=f"{int(step_s)}s")

    players = df[player_col].dropna().unique().tolist()
    nF, nP = len(frames), len(players)

    pos = np.full((nF, nP, 2), np.nan, dtype=float)
    status = np.full((nF, nP), "", dtype=object)

    for j, pid in enumerate(players):
        g = df[df[player_col] == pid].dropna(subset=["_ts"])
        if g.empty:
            continue

        # index by time, with status
        s = g.set_index("_ts")[[x_col, y_col, status_col]].sort_index()

        # pad to global frames
        r = s.reindex(frames, method="pad")

        # age since last *real* sample -> blank if too old
        last_real = s.index.to_series().reindex(frames, method="pad")
        age = (
            frames.to_series().reset_index(drop=True)
            - last_real.reset_index(drop=True)
        ).dt.total_seconds().to_numpy()

        r.loc[age > float(pad_limit_s), [x_col, y_col]] = np.nan

        pos[:, j, :] = r[[x_col, y_col]].to_numpy()
        status[:, j] = r[status_col].to_numpy()

    return frames.to_numpy(), players, pos, status


def animate_players_with_status(
    df_xy, pitch_xy,
    player_col="player_name", x_col="x_m", y_col="y_m",
    status_col="player_status",
    step_s=1, pad_limit_s=2, margin_m=6.0, fps=25,
    active_depth_m=None,   # same as in label_active_players
    rosenborg=None, gnum=None,  # match info for title
):
    """
    Returns matplotlib.animation.FuncAnimation.

    - Global clock from earliest to latest sample across all players.
    - Shows HH:MM:SS clock.
    - Colors players with stable colors.
    - Draws `player_status` (e.g. 'active' / 'bench') next to each dot.
    - Colors the active zone (depth >= active_depth_m) in light green, bench strip red.
    - If `rosenborg` and `gnum` are provided, uses that row to create a match title.
    """
    frames, players, pos, status = _prep_positions_with_status(
        df_xy, player_col, x_col, y_col, status_col, step_s, pad_limit_s
    )

    nF, nP = pos.shape[0], pos.shape[1]
    if nP == 0 or nF == 0:
        raise ValueError("No frames or players to animate (check your df_xy).")

    fig, ax = plt.subplots(figsize=(10, 6))

    # ---- build title from rosenborg & gnum, if provided ----
    if rosenborg is not None and gnum is not None:
        try:
            row = rosenborg.iloc[gnum - 1]  # gnum is 1-based in your code
            date_val = row.get("date", "")
            time_val = row.get("time", "")
            home_val = row.get("home", "")
            away_val = row.get("away", "")
            score_val = row.get("score", "")

            # Try to make a nice date/time string
            try:
                dt_str = pd.to_datetime(
                    f"{date_val} {time_val}"
                ).strftime("%Y-%m-%d %H:%M")
            except Exception:
                dt_str = f"{date_val} {time_val}"

            title_str = f"{dt_str} — {home_val} vs {away_val} ({score_val})"
            fig.suptitle(title_str, fontsize=14, y=0.99)
        except Exception:
            # fall back silently if anything goes wrong
            pass

    draw_pitch(
        ax, pitch_xy,
        margin_m=margin_m,
        show_outer_box=True,
        active_depth_m=active_depth_m,   # this controls red/green zones
        show_active_zone=True,
        show_scale_bar=True,
        scale_bar_length_m=10.0,
    )

    # Stable per-player colors
    cmap = plt.get_cmap("tab20").colors
    colors = [cmap[i % len(cmap)] for i in range(nP)]

    # Scatter: start with NaNs but correct number of points for colors
    offsets_init = np.full((nP, 2), np.nan)
    scat = ax.scatter(
        offsets_init[:, 0], offsets_init[:, 1],
        s=40, c=colors, edgecolors="black", linewidths=0.6, zorder=3
    )

    # Clock in axes coordinates (top centre, just under suptitle)
    clock_txt = ax.text(
        0.5, 1.02, "",
        transform=ax.transAxes,
        ha="center", va="bottom",
        fontsize=14, family="monospace",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.8),
        zorder=10,
        clip_on=False
    )

    # One text artist per player for status ("active"/"bench")
    status_texts = []
    for j in range(nP):
        txt = ax.text(
            0, 0, "",
            ha="center", va="bottom",
            fontsize=8,
            color="black",
            zorder=4,
            clip_on=True
        )
        status_texts.append(txt)

    def init():
        scat.set_offsets(offsets_init)
        clock_txt.set_text("")
        for txt in status_texts:
            txt.set_text("")
            txt.set_visible(False)
        return [scat, clock_txt, *status_texts]

    def update(i):
        XY = pos[i]  # (nP, 2)
        scat.set_offsets(XY)

        # update clock
        clock_txt.set_text(pd.to_datetime(frames[i]).strftime("%H:%M:%S"))

        # update per-player status labels
        for j in range(nP):
            x, y = XY[j]
            if np.isnan(x) or np.isnan(y):
                status_texts[j].set_visible(False)
                continue

            status_texts[j].set_visible(True)
            # small offset in y so text is above the dot
            status_texts[j].set_position((x, y + 0.8))

            lab = status[i, j]
            status_texts[j].set_text("" if (lab is None or lab != lab) else str(lab))

        return [scat, clock_txt, *status_texts]

    ani = animation.FuncAnimation(
        fig, update, init_func=init,
        frames=nF, interval=1000 / fps,
        blit=False  # keep False so text & clock behave nicely
    )

    plt.close(fig)  # avoid static figure under the JSHTML animation

    return ani

In [ ]:
ani = animate_players_with_status(
    df_labeled, pitch_xy,
    player_col="player_name",
    x_col="x_m", y_col="y_m",
    status_col="player_status",
    step_s=30,
    pad_limit_s=2,
    margin_m=6.0,
    fps=25,
    active_depth_m=active_depth,
    rosenborg=rosenborg,
    gnum=gnum,
)

ani  # last line in the cell

### Filter Out Inactive Players + Outside Game-time Data

In [ ]:
def _parse_time_of_day(value):
    """
    Robustly parse a time-of-day from various formats into a Timedelta since midnight.

    Handles examples like:
      - '18:00'
      - '18:00:00'
      - '17:22:09.7'
      - Timestamp('2024-05-10 18:00:00')
      - '2024-05-10 18:00:00'
    """
    # First try: interpret as full datetime and strip date
    try:
        ts = pd.to_datetime(value)
        # if this works, just take the time-of-day
        return pd.to_timedelta(ts.strftime("%H:%M:%S.%f"))
    except Exception:
        pass

    # Fallback: work with string directly
    s = str(value).strip()

    # If it looks like "YYYY-MM-DD HH:MM:SS", keep only the last piece
    if " " in s:
        s = s.split()[-1]

    # If it's just "HH:MM", add seconds
    if s.count(":") == 1:
        s = s + ":00"

    # Let pandas handle fractional seconds etc.
    return pd.to_timedelta(s)


def build_active_match_paths(
    df_labeled,
    rosenborg,
    gnum,
    player_col="player_name",
    status_col="player_status",
    min_active_s=300.0,   # minimum continuous active duration (5 minutes)
    max_gap_s=2.0,        # max allowed time gap within a continuous path (seconds)
):
    """
    Build a dataframe of continuous active paths using ONLY time-of-day.

    Steps:
      1) Use rosenborg[gnum-1]['time'] as kickoff (time-of-day only).
      2) Convert df_labeled['time'] to a Timedelta since midnight.
      3) Keep only rows in:
           - first half:  [kickoff, kickoff + 45 min)
           - second half: [kickoff + 60 min, kickoff + 105 min)
      4) Keep only rows where status_col == "active".
      5) For each player, split into paths by time continuity:
           - new path when gap > max_gap_s.
      6) Keep only paths with continuous active duration >= min_active_s.
      7) Return df with a 'path_id' column.
    """

    # ---- basic checks ----
    if player_col not in df_labeled.columns:
        raise ValueError(f"df_labeled must contain '{player_col}'.")
    if status_col not in df_labeled.columns:
        raise ValueError(f"df_labeled must contain '{status_col}' (e.g. 'player_status').")
    if "time" not in df_labeled.columns:
        raise ValueError("df_labeled must contain a 'time' column with time-of-day strings.")

    df = df_labeled.copy()

    # ---- parse time-of-day for df_labeled ----
    # '17:22:09.7' -> Timedelta('0 days 17:22:09.700000000')
    df["time_td"] = pd.to_timedelta(df["time"].astype(str))
    time_col = "time_td"
    t = df[time_col]

    # ---- get kickoff as time-of-day from rosenborg (robustly) ----
    row = rosenborg.iloc[gnum - 1]  # gnum is 1-based
    kickoff_td = _parse_time_of_day(row["time"])

    # ---- define first & second half windows (time-of-day only) ----
    first_start  = kickoff_td
    first_end    = kickoff_td + pd.Timedelta(minutes=45)

    second_start = kickoff_td + pd.Timedelta(minutes=60)   # 45 + 15
    second_end   = kickoff_td + pd.Timedelta(minutes=105)  # 45 + 15 + 45

    mask_first  = (t >= first_start) & (t < first_end)
    mask_second = (t >= second_start) & (t < second_end)
    mask_halves = mask_first | mask_second

    df = df[mask_halves].copy()
    if df.empty:
        raise ValueError(
            "No data within the specified match halves (by time-of-day).\n"
            f"Kickoff (time) = {row['time']}, "
            f"time range in df_labeled['time'] = [{df_labeled['time'].min()}, {df_labeled['time'].max()}]"
        )

    # tag which half (1 or 2)
    df["half"] = np.where(mask_first.loc[df.index], 1,
                          np.where(mask_second.loc[df.index], 2, np.nan))

    # ---- keep only rows where player is active ----
    df = df[df[status_col] == "active"].copy()
    if df.empty:
        raise ValueError("No active rows within match halves (time-of-day).")

    # ---- sort by player + time-of-day ----
    df = df.sort_values([player_col, time_col])

    # ---- assign path_ids per player based on time continuity ----
    df["path_id"] = -1
    global_path_counter = 0

    for pid, g in df.groupby(player_col, sort=False):
        times = g[time_col].values.astype("timedelta64[ns]")
        n = len(g)
        path_ids = np.full(n, -1, dtype=int)

        prev_idx = None
        for k in range(n):
            if prev_idx is None:
                # start a new path
                global_path_counter += 1
                path_ids[k] = global_path_counter
            else:
                dt = (times[k] - times[prev_idx]) / np.timedelta64(1, "s")
                if dt <= max_gap_s:
                    # same path (continuous in time)
                    path_ids[k] = global_path_counter
                else:
                    # gap too big -> new path
                    global_path_counter += 1
                    path_ids[k] = global_path_counter

            prev_idx = k

        df.loc[g.index, "path_id"] = path_ids

    # ---- compute duration of each path (using time-of-day) ----
    durations = (
        df.groupby("path_id")[time_col]
          .agg(["min", "max"])
          .rename(columns={"min": "t_start", "max": "t_end"})
    )
    durations["duration_s"] = (
        (durations["t_end"] - durations["t_start"])
        / np.timedelta64(1, "s")
    )

    # keep only paths with continuous active duration >= min_active_s
    valid_paths = durations[durations["duration_s"] >= float(min_active_s)].index

    df_paths = df[df["path_id"].isin(valid_paths)].copy()

    return df_paths


In [ ]:
min_active = 5 * 60   # 5 minutes continuous active
max_gap    = 2.0      # seconds between samples for continuity

df_paths = build_active_match_paths(
    df_labeled,
    rosenborg,
    gnum,
    player_col="player_name",
    status_col="player_status",
    min_active_s=min_active,
    max_gap_s=max_gap,
)

df_paths[["player_name", "half", "time", "time_td", "path_id"]]

### Player Dynamics in Pitch FOR

In [ ]:
def compute_msd_per_player(
    df_paths,
    player_col="player_name",
    x_col="x_m",
    y_col="y_m",
    time_col=None,      # "time_td" if present; otherwise we'll parse "time"
    path_col="path_id",
    max_lag_s=None      # optional: max time lag (seconds); None = as long as paths allow
):
    """
    Compute per-player MSD(tau) from df_paths, respecting path boundaries.

    Uses a time-averaged MSD:

      For each player and lag step k, we use ALL pairs (i, i+k) within each
      continuous path, never mixing across paths.

      MSD(k) = mean_i [ (x_{i+k} - x_i)^2 + (y_{i+k} - y_i)^2 ]
      tau_s(k) = mean_i [ t_{i+k} - t_i ]

    Parameters
    ----------
    df_paths : DataFrame
        Output of build_active_match_paths, containing at least:
        - player_col
        - x_col, y_col
        - path_col (continuous segments per player)
        - time_col (Timedelta) or 'time' as time-of-day string.

    Returns
    -------
    msd_df : DataFrame with columns:
        - player_col      (e.g. "player_name")
        - lag_step        (integer lag index k)
        - tau_s           (average time lag in seconds for this k)
        - MSD             (mean squared displacement in m^2)
        - N_pairs         (number of pairs contributing to this MSD)
    """
    if player_col not in df_paths.columns:
        raise ValueError(f"df_paths must contain '{player_col}'.")
    if path_col not in df_paths.columns:
        raise ValueError(f"df_paths must contain '{path_col}'.")

    df = df_paths.copy()

    # --- decide which time column to use ---
    if time_col is None:
        if "time_td" in df.columns:
            time_col = "time_td"
        elif "time" in df.columns:
            time_col = "time_td"
            df[time_col] = pd.to_timedelta(df["time"].astype(str))
        else:
            raise ValueError("df_paths must contain 'time_td' or 'time' for time information.")
    else:
        df[time_col] = pd.to_timedelta(df[time_col])

    # ensure positions are float
    df[x_col] = df[x_col].astype(float)
    df[y_col] = df[y_col].astype(float)

    msd_rows = []

    # --- group by player ---
    for pid, df_p in df.groupby(player_col, sort=False):
        # stats[lag_step] = [sum_sq, sum_tau, count]
        stats = {}

        # group by path to avoid cross-path displacements
        for path_id, g in df_p.groupby(path_col, sort=False):
            g = g.sort_values(time_col)
            if len(g) < 2:
                continue

            t = g[time_col].values.astype("timedelta64[ns]").astype("int64") / 1e9  # seconds
            x = g[x_col].to_numpy()
            y = g[y_col].to_numpy()
            n = len(g)

            # estimate typical dt in this path
            dt = np.median(np.diff(t))
            if dt <= 0 or np.isnan(dt):
                continue

            if max_lag_s is not None:
                max_k = int(max_lag_s // dt)
                max_k = max(1, min(max_k, n - 1))
            else:
                max_k = n - 1

            # for each lag step k, use ALL pairs (i, i+k) in this path
            for k in range(1, max_k + 1):
                dx = x[k:] - x[:-k]
                dy = y[k:] - y[:-k]
                if dx.size == 0:
                    continue

                sq = dx * dx + dy * dy
                tau = t[k:] - t[:-k]  # actual time lags for this k

                sum_sq = float(sq.sum())
                sum_tau = float(tau.sum())
                count = int(sq.size)

                if k not in stats:
                    stats[k] = [sum_sq, sum_tau, count]
                else:
                    stats[k][0] += sum_sq
                    stats[k][1] += sum_tau
                    stats[k][2] += count

        # build rows for this player
        for k in sorted(stats.keys()):
            sum_sq, sum_tau, count = stats[k]
            if count <= 0:
                continue
            msd_rows.append({
                player_col: pid,
                "lag_step": k,
                "tau_s": sum_tau / count,
                "MSD": sum_sq / count,
                "N_pairs": count,
            })

    msd_df = pd.DataFrame(msd_rows)

    return msd_df

In [ ]:
msd_df = compute_msd_per_player(
    df_paths,
    player_col="player_name",
    x_col="x_m",
    y_col="y_m",
    time_col=None,    # auto-uses time_td or parses time
    path_col="path_id",
    max_lag_s=None    # or e.g. 600 for max 10-min lag
)

msd_df

In [ ]:
def compute_step_lengths(
    df_paths,
    player_col="player_name",
    path_col="path_id",
    x_col="x_m",
    y_col="y_m",
    time_col=None,       # "time_td" if present; otherwise we'll parse "time"
    theta_deg=30.0,      # <<< angle threshold in degrees (you can change this)
):
    """
    Segment each player's paths into 'steps' based on turning-angle threshold.

    A step = straight-ish path segment such that when the local direction
    changes by more than theta_deg relative to the previous direction,
    we close the current step and start a new one.

    Returns a DataFrame with columns:
      - player_col
      - path_col
      - step_length (in metres)
    """
    if player_col not in df_paths.columns:
        raise ValueError(f"df_paths must contain '{player_col}'.")
    if path_col not in df_paths.columns:
        raise ValueError(f"df_paths must contain '{path_col}'.")
    if x_col not in df_paths.columns or y_col not in df_paths.columns:
        raise ValueError(f"df_paths must contain '{x_col}' and '{y_col}'.")

    df = df_paths.copy()

    # choose time column for sorting
    if time_col is None:
        if "time_td" in df.columns:
            time_col = "time_td"
        elif "time" in df.columns:
            time_col = "time_td"
            df[time_col] = pd.to_timedelta(df["time"].astype(str))
        else:
            raise ValueError("df_paths must contain 'time_td' or 'time' for time information.")
    else:
        df[time_col] = pd.to_timedelta(df[time_col])

    theta_rad = np.deg2rad(theta_deg)

    step_rows = []

    # work per player & path to respect path boundaries
    df = df.sort_values([player_col, path_col, time_col])

    for (pid, pth), g in df.groupby([player_col, path_col], sort=False):
        coords = g[[x_col, y_col]].to_numpy(dtype=float)
        n = len(coords)
        if n < 2:
            continue

        curr_start = 0          # index where current step starts
        prev_vec = None         # direction of previous move

        for i in range(1, n):
            v = coords[i] - coords[i-1]
            nv = np.linalg.norm(v)
            if nv == 0:
                # no movement, skip angle update
                continue

            if prev_vec is None:
                prev_vec = v
                continue

            # angle between previous direction and current direction
            dot = float(np.dot(prev_vec, v))
            denom = np.linalg.norm(prev_vec) * nv
            if denom == 0:
                angle = 0.0
            else:
                cosang = np.clip(dot / denom, -1.0, 1.0)
                angle = np.arccos(cosang)

            if angle > theta_rad:
                # close current step from curr_start to i-1
                step_len = np.linalg.norm(coords[i-1] - coords[curr_start])
                if step_len > 0:
                    step_rows.append({
                        player_col: pid,
                        path_col: pth,
                        "step_length": step_len,
                    })
                # start new step at i-1
                curr_start = i - 1
                prev_vec = v
            else:
                # continue current straight segment; update direction
                prev_vec = v

        # close final step to last point
        if curr_start < n-1:
            step_len = np.linalg.norm(coords[-1] - coords[curr_start])
            if step_len > 0:
                step_rows.append({
                    player_col: pid,
                    path_col: pth,
                    "step_length": step_len,
                })

    step_df = pd.DataFrame(step_rows)
    return step_df


In [ ]:
theta_deg = 35.0  # <<< you can tune this
step_df = compute_step_lengths(
    df_paths,
    player_col="player_name",
    path_col="path_id",
    x_col="x_m",
    y_col="y_m",
    theta_deg=theta_deg,
)

In [ ]:
def power_func(x,a,n):
    return a*(x**n)

In [ ]:
# 2) plotting loop
min_pairs = 20   # only show MSD points with at least this many pairs

for pid, g_msd in msd_df.groupby("player_name"):
    g_msd = g_msd[g_msd["N_pairs"] >= min_pairs].sort_values("tau_s")
    if g_msd.empty:
        continue

    steps = step_df.loc[step_df["player_name"] == pid, "step_length"].to_numpy()
    # filter out any non-positive values just in case
    steps = steps[steps > 0]
    if steps.size == 0:
        continue

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # ---- 1) MSD vs tau (linear) ----
    ax = axes[0]
    ax.loglog(g_msd["tau_s"], g_msd["MSD"],'k.')
    ax.loglog(g_msd["tau_s"][3:8], power_func(g_msd["tau_s"][3:8], 10, 1.5), 'k')
    ax.set_xlabel("Lag Time [s]")
    ax.set_ylabel("MSD [$m^{2}$]")

    # ---- 2) Step-size histogram (counts, linear axes) ----
    ax = axes[1]
    ax.hist(steps, bins=30, color = "r", edgecolor='black', linewidth=1.2)
    ax.set_xlabel("Step length, $l$ [m]")
    ax.set_ylabel("Count")

    # ---- 3) Step-size PDF, log-log ----
    ax = axes[2]

    # choose log-spaced bins between min and max step size
    smin, smax = steps.min(), steps.max()
    # safety: avoid crazy ranges
    if smin <= 0 or smax <= smin:
        plt.close(fig)
        continue

    n_bins = 40
    bins = np.logspace(np.log10(smin), np.log10(smax), n_bins + 1)

    counts, edges = np.histogram(steps, bins=bins)
    bin_centers = np.sqrt(edges[:-1] * edges[1:])  # geometric mean
    widths = np.diff(edges)
    N = steps.size

    # PDF estimate: p(l) ≈ counts / (N * width)
    pdf = counts / (N * widths)

    # keep only positive pdf values for log-log plotting
    mask = (pdf > 0) & (bin_centers > 0)
    bin_centers = bin_centers[mask]
    pdf = pdf[mask]

    ax.plot(bin_centers, pdf, 'r.')
    ax.loglog(bin_centers[13:26], power_func(bin_centers[13:26], 0.4, -1), 'k')
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Step length, $l$ [m]")
    ax.set_ylabel(r"$P \ (l)$")
    plt.tight_layout()
    plt.show()

In [ ]:
def build_step_heading_turn_df(
    df_paths,
    player_col="player_name",
    path_col="path_id",
    x_col="x_m",
    y_col="y_m",
    time_col=None,        # "time_td" if present; otherwise we'll parse "time"
    theta_deg=30.0,       # direction-change threshold for defining runs
):
    """
    Build:
      1) Run-level steps (heading + length) using a theta_deg cutoff.
      2) Micro turning angles between consecutive displacement vectors
         (frame-to-frame), unaffected by theta_deg.

    Returns
    -------
    step_df : DataFrame
        Columns:
          - player_col, path_col
          - step_index          (0,1,2,...) within each player/path
          - step_start_idx      (index of first sample of run)
          - step_end_idx        (index of last sample of run)
          - step_length         (metres)
          - heading_rad         (absolute heading of run, radians)
          - heading_deg         (degrees)
          - heading_weight      (= step_length; use for length-weighted roses)
    micro_turn_df : DataFrame
        Columns:
          - player_col, path_col
          - turning_angle_rad   (micro Δθ between v[i] and v[i+1], radians)
          - turning_angle_deg   (degrees)
    """
    df = df_paths.copy()

    # --- choose / prepare time column ---
    if time_col is None:
        if "time_td" in df.columns:
            time_col = "time_td"
        elif "time" in df.columns:
            time_col = "time_td"
            df[time_col] = pd.to_timedelta(df["time"].astype(str))
        else:
            raise ValueError("df_paths must contain 'time_td' or 'time'.")
    else:
        df[time_col] = pd.to_timedelta(df[time_col])

    theta_rad = np.deg2rad(theta_deg)

    # sort globally so each (player, path) is in time order
    df = df.sort_values([player_col, path_col, time_col])

    run_rows   = []
    turn_rows  = []

    for (pid, pth), g in df.groupby([player_col, path_col], sort=False):
        coords = g[[x_col, y_col]].to_numpy(dtype=float)
        idxs   = g.index.to_numpy()
        n = len(coords)
        if n < 2:
            continue

        # ---------- 1) RUN-BASED STEPS (for headings) ----------
        curr_start = 0
        prev_vec   = None
        steps      = []   # (start_i, end_i, step_length, heading_rad)

        for i in range(1, n):
            v  = coords[i] - coords[i-1]
            nv = np.linalg.norm(v)
            if nv == 0:
                continue

            if prev_vec is None:
                prev_vec = v
                continue

            dot   = float(np.dot(prev_vec, v))
            denom = np.linalg.norm(prev_vec) * nv
            if denom == 0:
                angle_change = 0.0
            else:
                cosang = np.clip(dot / denom, -1.0, 1.0)
                angle_change = np.arccos(cosang)

            if angle_change > theta_rad:
                # close current run [curr_start, i-1]
                end_i    = i - 1
                step_vec = coords[end_i] - coords[curr_start]
                step_len = np.linalg.norm(step_vec)
                if step_len > 0:
                    heading_rad = np.arctan2(step_vec[1], step_vec[0])
                    steps.append((curr_start, end_i, step_len, heading_rad))
                curr_start = i - 1
                prev_vec   = v
            else:
                prev_vec = v

        # close final run to last point
        if curr_start < n - 1:
            end_i    = n - 1
            step_vec = coords[end_i] - coords[curr_start]
            step_len = np.linalg.norm(step_vec)
            if step_len > 0:
                heading_rad = np.arctan2(step_vec[1], step_vec[0])
                steps.append((curr_start, end_i, step_len, heading_rad))

        for j, (s_i, e_i, step_len, heading_rad) in enumerate(steps):
            run_rows.append({
                player_col:      pid,
                path_col:        pth,
                "step_index":    j,
                "step_start_idx": int(idxs[s_i]),
                "step_end_idx":   int(idxs[e_i]),
                "step_length":    float(step_len),
                "heading_rad":    float(heading_rad),
                "heading_deg":    float(np.degrees(heading_rad)),
                "heading_weight": float(step_len),
            })

        # ---------- 2) MICRO TURNING ANGLES (frame-to-frame) ----------
        if n < 3:
            continue

        v  = coords[1:] - coords[:-1]         # micro-step vectors
        nv = np.linalg.norm(v, axis=1)

        for i in range(len(v) - 1):
            v1, v2 = v[i], v[i+1]
            if nv[i] == 0 or nv[i+1] == 0:
                continue

            dot   = float(np.dot(v1, v2))
            denom = nv[i] * nv[i+1]
            cosang = np.clip(dot / denom, -1.0, 1.0)
            dtheta = np.arccos(cosang)

            # sign from 2D cross product
            cross = v1[0]*v2[1] - v1[1]*v2[0]
            if cross < 0:
                dtheta = -dtheta

            dtheta = (dtheta + np.pi) % (2*np.pi) - np.pi

            turn_rows.append({
                player_col:        pid,
                path_col:          pth,
                "turning_angle_rad": float(dtheta),
                "turning_angle_deg": float(np.degrees(dtheta)),
            })

    step_df      = pd.DataFrame(run_rows)
    micro_turn_df = pd.DataFrame(turn_rows)
    return step_df, micro_turn_df

In [ ]:
step_seg_df, micro_turn_df = build_step_heading_turn_df(
    df_paths,
    player_col="player_name",
    path_col="path_id",
    x_col="x_m",
    y_col="y_m",
    theta_deg=30.0,
)

In [ ]:
player_col      = "player_name"
n_angle_bins    = 24      # sectors for rose plots
min_step_length = 0.0     # e.g. 1.0 or 2.0 to drop tiny runs from weighted plot

def rose_hist(ax, angles_rad, n_bins, weights=None, title=""):
    """
    Simple rose (circular histogram).
    angles_rad in [-pi, pi].
    """
    if angles_rad.size == 0:
        return

    counts, edges = np.histogram(
        angles_rad,
        bins=n_bins,
        range=(-np.pi, np.pi),
        weights=weights,
    )
    widths  = np.diff(edges)
    centers = edges[:-1] + widths / 2

    ax.bar(centers, counts, width=widths, bottom=0.0, align="center", alpha = 1, edgecolor = 'k')

    # 0 rad at +x (right goal), angles increase anticlockwise
    ax.set_theta_zero_location("E")
    ax.set_theta_direction(1)

    # ---- angular labels in radians (π notation) ----
    angle_degs  = [0, 90, 180, 270]
    angle_lbls  = [r"$0$", r"$\pi/2$", r"$\pi$", r"$3\pi/2$"]
    ax.set_thetagrids(angle_degs, angle_lbls)

    # ---- radial grid: 3 equally spaced circles, outer at edge, with rounded labels ----
    max_r = counts.max() if counts.size else 0
    if max_r > 0:
        ax.set_rlim(0, max_r)
        rticks = [max_r/3, 2*max_r/3, max_r]
        ax.set_rticks(rticks)
        rlabels = [str(int(round(v))) for v in rticks]
        ax.set_yticklabels(rlabels)

    ax.set_title(title)


for pid in step_seg_df[player_col].unique():
    # --- run-based headings for this player ---
    g_head = step_seg_df[step_seg_df[player_col] == pid]
    if g_head.empty:
        continue

    headings = g_head["heading_rad"].to_numpy()
    lengths  = g_head["step_length"].to_numpy()   # same as heading_weight

    # for weighted plot, optionally ignore very short runs
    mask_w      = lengths >= min_step_length
    headings_w  = headings[mask_w]
    lengths_w   = lengths[mask_w]

    # --- micro turning angles for this player (in radians) ---
    g_turn      = micro_turn_df[micro_turn_df[player_col] == pid]
    turning_rad = g_turn["turning_angle_rad"].to_numpy()
    if turning_rad.size == 0:
        continue

    # --- figure & axes ---
    fig = plt.figure(figsize=(16, 4))

    ax1 = fig.add_subplot(1, 3, 1, projection="polar")
    ax2 = fig.add_subplot(1, 3, 2, projection="polar")
    ax3 = fig.add_subplot(1, 3, 3)

    # 1) Run headings (unweighted rose)
    rose_hist(ax1, headings, n_angle_bins, weights=None,
              title="Run headings (unweighted)")

    # 2) Run headings (length-weighted rose)
    rose_hist(ax2, headings_w, n_angle_bins, weights=lengths_w,
              title="Run headings (length-weighted)")

    # 3) Micro turning angles Δθ histogram in radians
    ax3.hist(turning_rad, bins=36, range=(-np.pi, np.pi),
             edgecolor="black", linewidth=1.0)
    ax3.set_xlabel(r"Turning Angle $\Delta\theta$ [rad]")
    ax3.set_ylabel("Count")

    # x-axis ticks at nice multiples of π
    xticks       = [-np.pi, -np.pi/2, 0, np.pi/2, np.pi]
    xtick_labels = [r"$-\pi$", r"$-\pi/2$", r"$0$", r"$\pi/2$", r"$\pi$"]
    ax3.set_xticks(xticks)
    ax3.set_xticklabels(xtick_labels)
    ax3.set_xlim(-np.pi, np.pi)

    plt.show()

### Team Dynamics in Pitch FOR

In [ ]:
def compute_team_centroid(
    df_paths,
    x_col="x_m",
    y_col="y_m",
    time_col=None,    # we'll auto-pick if None
    min_players=1,    # only keep times with at least this many players
):
    """
    Compute the team centroid (mean x,y) at each time from df_paths.

    Returns a DataFrame with columns:
      - time_col (e.g. 'time_td')
      - x_centroid
      - y_centroid
      - n_players   (number of players contributing at that time)
    """
    df = df_paths.copy()

    # ---- pick / build time column ----
    if time_col is None:
        if "time_td" in df.columns:
            time_col = "time_td"
        elif "time" in df.columns:
            time_col = "time_td"
            df[time_col] = pd.to_timedelta(df["time"].astype(str))
        else:
            raise ValueError("df_paths must contain 'time_td' or 'time'.")

    # keep just what we need
    df = df[[time_col, x_col, y_col]].dropna(subset=[time_col, x_col, y_col])

    # group by time and compute mean x,y and player count
    grp = (
        df.groupby(time_col)
          .agg(
              x_centroid=(x_col, "mean"),
              y_centroid=(y_col, "mean"),
              n_players=(x_col, "size"),
          )
          .reset_index()
    )

    # filter out times with too few players if desired
    if min_players > 1:
        grp = grp[grp["n_players"] >= min_players]

    # sort by time just to be safe
    grp = grp.sort_values(time_col).reset_index(drop=True)

    return grp

In [ ]:
centroid_df = compute_team_centroid(
    df_paths,
    x_col="x_m",
    y_col="y_m",
    # time_col=None -> will auto-use 'time_td' if present
    min_players=1,   # or e.g. 5, 8, 11 if you want a minimum
)

centroid_df.head()

In [ ]:
def animate_centroid(
    centroid_df,
    pitch_xy,
    x_col="x_centroid",
    y_col="y_centroid",
    time_col=None,   # auto-pick if None
    margin_m=6.0,
    fps=25,
    frame_step=1,    # <<< new: keep every Nth row to speed up animation
):
    """
    Animate the movement of the team centroid over time.

    Expects centroid_df to have:
      - x_col, y_col (e.g. 'x_centroid', 'y_centroid')
      - time_col (e.g. 'time_td' or 'time'/'timestamp')
      - optionally 'n_players'

    frame_step: keep every Nth row (1 = use all rows, 2 = every other row, ...)
    """
    df = centroid_df.copy().dropna(subset=[x_col, y_col])

    if df.empty:
        raise ValueError("centroid_df has no valid centroid positions.")

    # ---- choose / build time column ----
    if time_col is None:
        if "time_td" in df.columns:
            time_col = "time_td"
            time_series = df[time_col]
        elif "timestamp" in df.columns or "time" in df.columns:
            time_series = _choose_time(df)  # uses your helper
            time_col = "_ts"
            df[time_col] = time_series
        else:
            time_col = None
            time_series = df.index.to_series()
    else:
        if time_col not in df.columns:
            raise ValueError(f"{time_col!r} not in centroid_df columns.")
        col = df[time_col]
        if str(col.dtype).startswith("timedelta64"):
            time_series = col
        else:
            try:
                time_series = pd.to_timedelta(col.astype(str))
            except Exception:
                time_series = pd.to_datetime(col, errors="ignore")

    # ---- sort by time, then subsample rows with frame_step ----
    if time_col is not None:
        df = df.assign(_time=time_series).sort_values("_time")
        time_series = df["_time"]
    else:
        df = df.reset_index(drop=True)

    # keep every Nth row
    df = df.iloc[::frame_step].reset_index(drop=True)
    if hasattr(time_series, "iloc"):
        T = time_series.iloc[::frame_step].reset_index(drop=True)
    else:
        T = time_series[::frame_step]

    X = df[x_col].to_numpy()
    Y = df[y_col].to_numpy()
    n_frames = len(df)

    has_n_players = "n_players" in df.columns

    # ---- set up figure & pitch ----
    fig, ax = plt.subplots(figsize=(10, 6))
    draw_pitch(ax, pitch_xy, margin_m=margin_m)

    # centroid point
    scat = ax.scatter(
        [np.nan], [np.nan],
        s=70, c="red", edgecolors="black",
        linewidths=0.8, zorder=4
    )

    # centroid path
    line, = ax.plot([], [], lw=1.5, color="red", alpha=0.7, zorder=3)

    # clock text above pitch
    clock_txt = ax.text(
        0.5, 1.02, "",
        transform=ax.transAxes,
        ha="center", va="bottom",
        fontsize=14, family="monospace",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.8),
        zorder=10,
        clip_on=False,
    )

    # number of players text (if available)
    if has_n_players:
        n_txt = ax.text(
            0.01, 0.99, "",
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=10,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7),
            zorder=10,
            clip_on=False,
        )
    else:
        n_txt = None

    def _format_time(t):
        if isinstance(t, pd.Timedelta):
            sec = int(t.total_seconds())
            h = sec // 3600
            m = (sec % 3600) // 60
            s = sec % 60
            return f"{h:02d}:{m:02d}:{s:02d}"
        if isinstance(t, (pd.Timestamp, np.datetime64)):
            tt = pd.to_datetime(t)
            return tt.strftime("%H:%M:%S")
        return str(t)

    def init():
        scat.set_offsets([[np.nan, np.nan]])
        line.set_data([], [])
        clock_txt.set_text("")
        if n_txt is not None:
            n_txt.set_text("")
            return scat, line, clock_txt, n_txt
        return scat, line, clock_txt

    def update(i):
        cx, cy = X[i], Y[i]
        scat.set_offsets([[cx, cy]])
        line.set_data(X[:i+1], Y[:i+1])

        ti = T.iloc[i] if hasattr(T, "iloc") else T[i]
        clock_txt.set_text(_format_time(ti))

        if n_txt is not None:
            n_val = int(df["n_players"].iloc[i])
            n_txt.set_text(f"n = {n_val}")

        if n_txt is not None:
            return scat, line, clock_txt, n_txt
        return scat, line, clock_txt

    ani = animation.FuncAnimation(
        fig,
        update,
        init_func=init,
        frames=n_frames,
        interval=1000 / fps,
        blit=False,
    )

    plt.close(fig)
    return ani

In [ ]:
ani_centroid = animate_centroid(
    centroid_df,
    pitch_xy,
    x_col="x_centroid",
    y_col="y_centroid",
    margin_m=6.0,
    fps=25,
    frame_step=30,   # show every 10th centroid sample
)

ani_centroid

In [ ]:
def compute_centroid_msd(
    centroid_df,
    x_col="x_centroid",
    y_col="y_centroid",
    time_col=None,   # auto-choose 'time_td' or 'time'
):
    """
    Compute MSD for the team centroid trajectory, using all pairs
    separated by k time steps (like we did for each player).

    Returns a DataFrame with:
      - tau_s   : mean lag time [s] for that step-k
      - MSD     : mean squared displacement [m^2]
      - N_pairs : number of pairs contributing
    """
    df = centroid_df.copy()

    # ---- choose / build time column ----
    if time_col is None:
        if "time_td" in df.columns:
            time_col = "time_td"
        elif "time" in df.columns:
            time_col = "time_td"
            df[time_col] = pd.to_timedelta(df["time"].astype(str))
        else:
            raise ValueError("centroid_df must contain 'time_td' or 'time'.")

    # ensure sorted by time
    df = df.sort_values(time_col).dropna(subset=[time_col, x_col, y_col])
    t = df[time_col].dt.total_seconds().to_numpy()
    x = df[x_col].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)

    N = len(df)
    if N < 2:
        raise ValueError("Not enough centroid samples to compute MSD.")

    max_step = N - 1

    tau_s   = np.empty(max_step, dtype=float)
    msd     = np.empty(max_step, dtype=float)
    n_pairs = np.empty(max_step, dtype=int)

    for k in range(1, max_step + 1):
        # displacements over k steps
        dx = x[k:] - x[:-k]
        dy = y[k:] - y[:-k]
        dr2 = dx*dx + dy*dy

        # lag times (actual, in case of slight irregular sampling)
        dt_k = t[k:] - t[:-k]

        tau_s[k-1]   = dt_k.mean()
        msd[k-1]     = dr2.mean()
        n_pairs[k-1] = dr2.size

    msd_df = pd.DataFrame({
        "tau_s": tau_s,
        "MSD": msd,
        "N_pairs": n_pairs,
    })
    return msd_df


In [ ]:
centroid_df = compute_team_centroid(df_paths, x_col="x_m", y_col="y_m")

msd_centroid_df = compute_centroid_msd(
    centroid_df,
    x_col="x_centroid",
    y_col="y_centroid",
)

msd_centroid_df.head()


In [ ]:
def compute_centroid_step_lengths(
    centroid_df,
    x_col="x_centroid",
    y_col="y_centroid",
    time_col=None,
    theta_deg=35.0,
):
    """
    Segment the centroid trajectory into straight-ish runs using a turning-angle
    cutoff theta_deg (in degrees), and compute step lengths (and headings).

    Returns a DataFrame with columns:
      - step_index
      - step_start_idx   (index in centroid_df)
      - step_end_idx
      - step_length      [m]
      - heading_rad      (run direction, radians)
      - heading_deg
    """
    df = centroid_df.copy().dropna(subset=[x_col, y_col])

    # --- choose / build time column for sorting ---
    if time_col is None:
        if "time_td" in df.columns:
            time_col = "time_td"
        elif "time" in df.columns:
            time_col = "time_td"
            df[time_col] = pd.to_timedelta(df["time"].astype(str))
        elif "timestamp" in df.columns:
            time_col = "timestamp"
            df[time_col] = pd.to_datetime(df["timestamp"], errors="coerce")
        else:
            # fall back to index order
            time_col = None

    if time_col is not None:
        df = df.sort_values(time_col)

    coords = df[[x_col, y_col]].to_numpy(dtype=float)
    idxs   = df.index.to_numpy()
    n = len(coords)
    if n < 2:
        raise ValueError("Not enough centroid samples to define steps.")

    theta_rad = np.deg2rad(theta_deg)

    rows = []
    curr_start = 0
    prev_vec = None

    for i in range(1, n):
        v  = coords[i] - coords[i - 1]
        nv = np.linalg.norm(v)
        if nv == 0:
            continue

        if prev_vec is None:
            prev_vec = v
            continue

        dot   = float(np.dot(prev_vec, v))
        denom = np.linalg.norm(prev_vec) * nv
        if denom == 0:
            angle_change = 0.0
        else:
            cosang = np.clip(dot / denom, -1.0, 1.0)
            angle_change = np.arccos(cosang)

        if angle_change > theta_rad:
            # close current run [curr_start, i-1]
            end_i    = i - 1
            step_vec = coords[end_i] - coords[curr_start]
            step_len = np.linalg.norm(step_vec)
            if step_len > 0:
                heading_rad = np.arctan2(step_vec[1], step_vec[0])
                rows.append({
                    "step_index":      len(rows),
                    "step_start_idx":  int(idxs[curr_start]),
                    "step_end_idx":    int(idxs[end_i]),
                    "step_length":     float(step_len),
                    "heading_rad":     float(heading_rad),
                    "heading_deg":     float(np.degrees(heading_rad)),
                })
            curr_start = i - 1
            prev_vec   = v
        else:
            prev_vec = v

    # close final run to last point
    if curr_start < n - 1:
        end_i    = n - 1
        step_vec = coords[end_i] - coords[curr_start]
        step_len = np.linalg.norm(step_vec)
        if step_len > 0:
            heading_rad = np.arctan2(step_vec[1], step_vec[0])
            rows.append({
                "step_index":      len(rows),
                "step_start_idx":  int(idxs[curr_start]),
                "step_end_idx":    int(idxs[end_i]),
                "step_length":     float(step_len),
                "heading_rad":     float(heading_rad),
                "heading_deg":     float(np.degrees(heading_rad)),
            })

    step_df = pd.DataFrame(rows)
    return step_df

In [ ]:
theta_deg = 35.0  # tune as you like
centroid_step_df = compute_centroid_step_lengths(
    centroid_df,
    x_col="x_centroid",
    y_col="y_centroid",
    theta_deg=theta_deg,
)
centroid_step_df.head()


In [ ]:
# --- parameters you can tweak ---
min_pairs   = 20   # only show MSD points with at least this many pairs
n_bins_hist = 30   # bins for raw step-length histogram
n_bins_pdf  = 40   # bins for log-binned PDF

# ---------- 1) MSD vs lag time (filter by N_pairs) ----------
g_msd = msd_centroid_df.copy()
g_msd = g_msd[g_msd["N_pairs"] >= min_pairs].sort_values("tau_s")

if g_msd.empty:
    raise ValueError("No MSD points with N_pairs >= min_pairs.")

# ---------- 2) Step-length array ----------
steps = centroid_step_df["step_length"].to_numpy(dtype=float)
steps = steps[steps > 0]
if steps.size == 0:
    raise ValueError("No positive step lengths in centroid_step_df.")

# ---------- 3) PDF of step lengths (log-binned) ----------
smin, smax = steps.min(), steps.max()
if smax <= smin:
    raise ValueError("Centroid step lengths have no spread.")

bins    = np.logspace(np.log10(smin), np.log10(smax), n_bins_pdf + 1)
counts, edges = np.histogram(steps, bins=bins)
widths  = np.diff(edges)
centers = np.sqrt(edges[:-1] * edges[1:])  # geometric mean of bin edges
N       = steps.size

pdf = counts / (N * widths)

mask    = (pdf > 0) & (centers > 0)
centers = centers[mask]
pdf     = pdf[mask]

# ---------- 4) Plotting ----------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) MSD vs lag time (log-log)
ax = axes[0]
ax.loglog(g_msd["tau_s"], g_msd["MSD"], "k.")
ax.loglog(g_msd["tau_s"][3:8], power_func(g_msd["tau_s"][3:8], 10, 1.5), 'k')
ax.set_xlabel("Lag time $\\tau$ [s]")
ax.set_ylabel("MSD [$\\mathrm{m}^2$]")

# (2) Histogram of step sizes (counts)
ax = axes[1]
ax.hist(steps, bins=n_bins_hist, edgecolor="black", color = "r",alpha = 0.7, linewidth=1.0)
ax.set_xlabel("Centroid step length, $\\ell$ [m]")
ax.set_ylabel("Count")

# (3) PDF of step sizes (log-log)
ax = axes[2]
ax.loglog(centers, pdf, ".", markerfacecolor = "k", markeredgecolor = "k")
ax.loglog(centers, power_func(centers, 0.1, -1), 'k')
ax.set_xlabel("Centroid step length, $\\ell$ [m]")
ax.set_ylabel("$P(\\ell)$")

plt.tight_layout()
plt.show()


### Player Dynamics in Centroid FOR.

In [ ]:
# ---- 1) ensure time_td exists in both dfs ----
def _ensure_time_td(df, time_col_guess=("time_td", "time", "timestamp")):
    df = df.copy()
    if "time_td" in df.columns:
        return df

    if "time" in df.columns:
        df["time_td"] = pd.to_timedelta(df["time"].astype(str))
        return df

    if "timestamp" in df.columns:
        # if timestamps have fake dates but correct times, this still works
        ts = pd.to_datetime(df["timestamp"], errors="coerce")
        # convert to time-of-day timedelta
        df["time_td"] = pd.to_timedelta(ts.dt.strftime("%H:%M:%S.%f"))
        return df

    raise ValueError("Need 'time_td' or 'time' or 'timestamp' in dataframe.")

dfp = _ensure_time_td(df_paths)
cend = _ensure_time_td(centroid_df)

# ---- 2) sort for asof-merge ----
dfp = dfp.sort_values("time_td")
cend = cend.sort_values("time_td")

# ---- 3) align centroid to each player sample ----
# tolerance controls max mismatch allowed (set to your sampling resolution)
tolerance_s = 0.6  # e.g. 0.6s for ~1 Hz data; increase if needed

df_rel = pd.merge_asof(
    dfp,
    cend[["time_td", "x_centroid", "y_centroid"]],
    on="time_td",
    direction="nearest",
    tolerance=pd.to_timedelta(tolerance_s, unit="s")
)

# If some rows didn't find a centroid within tolerance, they'll be NaN here:
df_rel = df_rel.dropna(subset=["x_centroid", "y_centroid"])

# ---- 4) compute centroid-frame coordinates ----
df_rel["x_rel"] = df_rel["x_m"] - df_rel["x_centroid"]
df_rel["y_rel"] = df_rel["y_m"] - df_rel["y_centroid"]

df_rel.head()

In [ ]:
msd_rel_df = compute_msd_per_player(
    df_rel,                      # df_rel has x_rel, y_rel
    player_col="player_name",
    x_col="x_rel",
    y_col="y_rel",
    time_col="time_td",          # or None if df_rel still has time/time_td
    path_col="path_id",
    max_lag_s=None
)

In [ ]:
theta_deg = 30.0  # or whatever you want

step_rel_df = compute_step_lengths(
    df_rel,
    player_col="player_name",
    path_col="path_id",
    x_col="x_rel",
    y_col="y_rel",
    time_col="time_td",   # or None if df_rel already has time_td/time
    theta_deg=theta_deg,
)

In [ ]:
# 2) plotting loop
min_pairs = 20   # only show MSD points with at least this many pairs

for pid, g_msd in msd_rel_df.groupby("player_name"):
    g_msd = g_msd[g_msd["N_pairs"] >= min_pairs].sort_values("tau_s")
    if g_msd.empty:
        continue

    steps = step_rel_df.loc[step_rel_df["player_name"] == pid, "step_length"].to_numpy()
    # filter out any non-positive values just in case
    steps = steps[steps > 0]
    if steps.size == 0:
        continue

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # ---- 1) MSD vs tau (linear) ----
    ax = axes[0]
    ax.loglog(g_msd["tau_s"], g_msd["MSD"],'k.')
    ax.loglog(g_msd["tau_s"][3:8], power_func(g_msd["tau_s"][3:8], 7, 1), 'k')
    ax.set_xlabel("Lag Time [s]")
    ax.set_ylabel("MSD [$m^{2}$]")

    # ---- 2) Step-size histogram (counts, linear axes) ----
    ax = axes[1]
    ax.hist(steps, bins=30, color = "r", edgecolor='black', linewidth=1.2)
    ax.set_xlabel("Step length, $l$ [m]")
    ax.set_ylabel("Count")

    # ---- 3) Step-size PDF, log-log ----
    ax = axes[2]

    # choose log-spaced bins between min and max step size
    smin, smax = steps.min(), steps.max()
    # safety: avoid crazy ranges
    if smin <= 0 or smax <= smin:
        plt.close(fig)
        continue

    n_bins = 40
    bins = np.logspace(np.log10(smin), np.log10(smax), n_bins + 1)

    counts, edges = np.histogram(steps, bins=bins)
    bin_centers = np.sqrt(edges[:-1] * edges[1:])  # geometric mean
    widths = np.diff(edges)
    N = steps.size

    # PDF estimate: p(l) ≈ counts / (N * width)
    pdf = counts / (N * widths)

    # keep only positive pdf values for log-log plotting
    mask = (pdf > 0) & (bin_centers > 0)
    bin_centers = bin_centers[mask]
    pdf = pdf[mask]

    ax.plot(bin_centers, pdf, 'r.')
    ax.loglog(bin_centers[13:26], power_func(bin_centers[13:26], 0.4, -1), 'k')
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Step length, $l$ [m]")
    ax.set_ylabel(r"$P \ (l)$")
    plt.tight_layout()
    plt.show()

### Computing Player Orientation

### Computing Team Orientation

### Flocking-Synchronising Behaviour

### Generative Modelling - Full Match Data?